# QuOpt-Agents — Revised Implementation (Response to Editors and Reviewers)

**Paper:** *Quantum-Enhanced Multi-Agent Framework for Portfolio Optimization*

This notebook replaces the original submission code. It is the single artifact that
regenerates **every number, table and figure** in the revised manuscript.

---

## Why the code was rewritten, not patched

An audit of the submitted notebook found four issues that made the reported results
impossible to defend. Each is fixed here.

| # | Problem in the submitted code | Consequence | Fix in this notebook |
|---|---|---|---|
| 1 | The "quantum" optimiser called `NumPyMinimumEigensolver`, an **exact classical eigensolver**. `QAOA`, `COBYLA` and `Sampler` were imported but never used. | The manuscript's QAOA description (depth, shots, optimiser) described code that did not exist. | Sections 5–6: a real, fully specified QAOA — variational parameters optimised by COBYLA, depth p=2, 1024 measurement shots, exact state-vector simulation, cross-validated against Qiskit Aer. |
| 2 | `_quantum_super_upgraded` **seeded itself with the greedy solution**, searched four risk factors, ran local search, then re-searched until it beat greedy by 1%, and returned the best Sharpe found. Greedy returned a single solution. | The proposed method was a *superset* search containing its own baseline. It could not lose. The 92%/98% win rates measured this asymmetry, not optimisation quality. | Section 9: Tier 2 is QAOA sampling only, with no baseline seeding and no target-chasing. All metaheuristics receive an **identical objective-evaluation budget**. |
| 3 | Portfolios were selected and scored on the **same 252-day window**. | Win rates were in-sample fit, not performance. | Sections 3 and 10: every synthetic scenario is split into estimation and evaluation windows; real data uses non-overlapping walk-forward windows. |
| 4 | No real market data, no GA/PSO/DE/SA, no ablation, no Wilcoxon test, no confidence intervals, no exhaustive optimum. | Tables 4, 8, 9 and the Wilcoxon statistics had **no code behind them**. | Sections 8, 10–13: all baselines, real Yahoo Finance walk-forward, both ablations, exhaustive enumeration, and the full statistical suite. |

## What the honest result now is

Exhaustive enumeration (Section 8) is tractable at the studied size —
C(16,5) = 4368 portfolios. It establishes the **exact optimum**, which lets us report an
optimality gap instead of a win rate. The finding this notebook supports is:

> At N=16, B=5 the cardinality-constrained problem is small enough that QAOA and strong
> classical search both reach the exact optimum. QAOA provides **no advantage** here; the
> contribution is the agent architecture, the reproducible evaluation protocol, and a
> quantified account of where the quantum layer begins to matter.

This is the claim Editor 1's Comment 2 requires, and it is the claim the code can prove.

## Reviewer comment to section map

| Reviewer point | Addressed in |
|---|---|
| E1-C1 abbreviations, presentation | Section 14 figures; manuscript text |
| E1-C2 what does QAOA contribute? | Section 8 (exact optimum), Section 11 (Tier-2 ablation), Section 12 (scale-up) |
| E1-C3 QAOA specification | Section 5 (config), Section 6 (simulator and selection rule) |
| E1-C4 reference recency | manuscript only |
| E2-C1 QUBO formulation error | Section 4, with an executable proof of correctness |
| E2-C2/C3 statistics, Wilcoxon, CIs | Section 13 |
| E2-C4 Figure 3 vanishing curve | Section 14, `figure_convergence` |
| E2-C5 baselines | Section 8 |
| E2-C6 ablation | Section 11 |
| E2-C7 reproducibility | this notebook plus Section 15 environment capture |


## 1. Environment and global configuration

Everything the manuscript needs to state about the computational setup is captured here
and printed at the end of the notebook (Section 15), so the "Reproducibility" paragraph
can be pasted directly from the output rather than written from memory.

Set `QUICK = True` for a fast structural check (a few minutes). Set `QUICK = False` to
regenerate the manuscript numbers.


In [ ]:
import itertools
import json
import math
import os
import platform
import sys
import time
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats as sps
from scipy.optimize import minimize

warnings.filterwarnings("ignore", category=FutureWarning)

QUICK = False

# Kaggle writes to /kaggle/working; locally we keep artefacts beside the notebook.
OUTDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
for sub in ("results", "figures", "tables", "cache"):
    (OUTDIR / sub).mkdir(parents=True, exist_ok=True)


@dataclass
class QAOAConfig:
    """Complete QAOA specification. Every field appears in manuscript Section 3.4."""
    depth: int = 2                      # p
    shots: int = 1024
    optimizer: str = "COBYLA"
    maxiter: int = 150
    n_restarts: int = 3                 # random restarts of the classical outer loop
    init_state: str = "uniform_superposition"
    mixer: str = "transverse_field_X"
    max_qubits: int = 22                # exact state-vector ceiling on 16 GB RAM
    seed: int = 1234


@dataclass
class Config:
    n_assets: int = 16
    budget: int = 5                     # B, cardinality constraint
    q_risk: float = 0.5                 # risk aversion of the QUBO surrogate
    seed: int = 42

    # Synthetic study: estimation window then a disjoint evaluation window.
    n_scenarios: int = 50
    estim_days: int = 252
    eval_days: int = 126

    # Real-data walk-forward study.
    tickers: Tuple[str, ...] = (
        "AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "JPM", "JNJ",
        "XOM", "PG", "UNH", "HD", "V", "MA", "KO", "PFE",
    )
    start: str = "2018-01-01"
    end: str = "2025-12-31"
    n_windows: int = 20
    lookback: int = 252
    horizon: int = 63

    # Identical objective-evaluation budget for every stochastic classical searcher.
    eval_budget: int = 3000

    # Scale-up sweep (N, B) for Section 12.
    scaleup: Tuple[Tuple[int, int], ...] = (
        (12, 4), (16, 5), (20, 6), (24, 6), (30, 8), (40, 10),
    )

    qaoa: QAOAConfig = field(default_factory=QAOAConfig)


CFG = Config()

if QUICK:
    CFG.n_scenarios = 6
    CFG.n_windows = 3
    CFG.eval_budget = 400
    CFG.scaleup = ((12, 4), (16, 5), (24, 6))
    CFG.qaoa.maxiter = 30
    CFG.qaoa.n_restarts = 1

RNG = np.random.default_rng(CFG.seed)

ENV = {
    "python": sys.version.split()[0],
    "platform": f"{platform.system()} {platform.release()} ({platform.machine()})",
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": __import__("scipy").__version__,
    "matplotlib": matplotlib.__version__,
    "cpu_count": os.cpu_count(),
}
print("Configuration")
print(f"  N={CFG.n_assets}, B={CFG.budget}, C(N,B)={math.comb(CFG.n_assets, CFG.budget):,}")
print(f"  QAOA: p={CFG.qaoa.depth}, shots={CFG.qaoa.shots}, "
      f"{CFG.qaoa.optimizer}(maxiter={CFG.qaoa.maxiter}), restarts={CFG.qaoa.n_restarts}")
print(f"  evaluation budget per classical searcher: {CFG.eval_budget:,}")
print(f"  QUICK={QUICK}, artefacts -> {OUTDIR.resolve()}")
print("\nEnvironment")
for k, v in ENV.items():
    print(f"  {k}: {v}")


## 2. The objective, and how search effort is counted

Two distinct functions were conflated in the submitted code. Separating them is what makes
the three-tier architecture meaningful rather than decorative.

**The true objective** `f(S)` is the Sharpe ratio of the equal-weighted portfolio holding
the `B` assets in `S`, computed on the estimation window:

    f(S) = (w' mu) / sqrt(w' Sigma w),   w_i = 1/B for i in S, else 0

This is the *only* quantity used to compare methods. It is a ratio, so it is not a
quadratic form and cannot be encoded in a QUBO.

**The quadratic surrogate** is the mean-variance QUBO of Section 4. QAOA optimises the
surrogate; its measurement outcomes are then scored by `f`. That gap between surrogate and
objective is precisely the role Tier 3 plays, and it is why the architecture has three
tiers instead of one.

**Evaluation accounting.** Every stochastic classical searcher is wrapped in an
`EvalCounter` and stopped at exactly `CFG.eval_budget` evaluations of `f`. This removes the
asymmetry that produced the original win rates: previously the proposed method explored
several hundred candidates while "greedy" explored one. The counter also records the
best-so-far trajectory, which is what Figure 3 plots (Editor 2, Comment 4).


In [ ]:
class BudgetExhausted(Exception):
    """Raised to unwind a searcher the moment its evaluation budget is spent."""


def portfolio_moments(mu: np.ndarray, cov: np.ndarray,
                      selection: Sequence[int]) -> Tuple[float, float]:
    """Annualised return and volatility of the equal-weighted portfolio on `selection`."""
    idx = np.asarray(sorted(selection), dtype=int)
    if idx.size == 0:
        return 0.0, 0.0
    w = np.zeros(mu.size)
    w[idx] = 1.0 / idx.size
    ret = float(w @ mu)
    var = float(w @ cov @ w)
    return ret, float(np.sqrt(max(var, 0.0)))


def sharpe(mu: np.ndarray, cov: np.ndarray, selection: Sequence[int],
           rf: float = 0.0) -> float:
    ret, vol = portfolio_moments(mu, cov, selection)
    return (ret - rf) / vol if vol > 1e-12 else -np.inf


class EvalCounter:
    """Wraps the objective so that every searcher is held to the same budget.

    Also records the best-so-far trajectory against cumulative evaluation count, which is
    the correct x-axis for a convergence plot comparing methods with different iteration
    semantics (a GA "generation" and an SA "step" are not comparable units).
    """

    def __init__(self, mu: np.ndarray, cov: np.ndarray, budget: int,
                 rf: float = 0.0) -> None:
        self.mu, self.cov, self.budget, self.rf = mu, cov, budget, rf
        self.n_evals = 0
        self.best_value = -np.inf
        self.best_selection: Tuple[int, ...] = ()
        self.trajectory: List[Tuple[int, float]] = []

    def __call__(self, selection: Sequence[int]) -> float:
        if self.n_evals >= self.budget:
            raise BudgetExhausted
        self.n_evals += 1
        value = sharpe(self.mu, self.cov, selection, self.rf)
        if value > self.best_value:
            self.best_value = value
            self.best_selection = tuple(sorted(selection))
            self.trajectory.append((self.n_evals, value))
        return value

    @property
    def remaining(self) -> int:
        return max(0, self.budget - self.n_evals)

    def curve(self, length: Optional[int] = None) -> np.ndarray:
        """Step-function expansion of the trajectory onto a dense evaluation grid.

        Figure 3 in the submission plotted only the improvement points, so a method that
        converged after a handful of evaluations rendered as a curve that appeared to
        vanish. Expanding to a step function fixes that.
        """
        length = length or self.budget
        curve = np.full(length, np.nan)
        for n_eval, value in self.trajectory:
            if n_eval <= length:
                curve[n_eval - 1] = value
        if np.all(np.isnan(curve)):
            return curve
        first = int(np.argmax(~np.isnan(curve)))
        curve[:first] = curve[first]
        return pd.Series(curve).ffill().to_numpy()


def top_b(values: np.ndarray, b: int) -> List[int]:
    """Indices of the b largest entries, ties broken by index for determinism."""
    order = np.lexsort((np.arange(values.size), -values))
    return sorted(int(i) for i in order[:b])


print("Objective and evaluation accounting ready.")


## 3. Agent 1 — Data Preparation

### 3.1 Synthetic multi-regime generator

Three corrections to the submitted generator:

**Correlation structure was inverted.** The submitted code set `correlation_bias` to 0.6
for bull and 0.3 for bear. The manuscript defines bear regimes as the high-correlation
state, which is also the financially correct behaviour — cross-sectional correlation spikes
in drawdowns. The values are now bull 0.30 and bear 0.60.

**Correlation matrices are now PSD by construction.** The submitted code drew a uniform
matrix, symmetrised it, added a scalar bias, clipped to `[-0.9, 0.9]`, then repaired it by
eigenvalue flooring and re-normalisation. The repair step changes the realised correlation
away from the nominal `beta`, so the regime parameter reported in the manuscript was not the
parameter actually simulated. A single-factor loading construction, `Corr_ij = a_i a_j` for
`i != j` with `a_i` centred on `sqrt(beta)`, is positive semi-definite with no repair and
has mean pairwise correlation `beta` by construction. The cell below asserts this.

**Regime counts are now fixed, not random.** The submitted code drew each scenario's regime
from `np.random.choice`, so the per-regime counts were an accident of the seed and did not
match the manuscript's Table 5. They are now fixed at 14 / 11 / 6 / 7 / 12 = 50.

### 3.2 Estimation and evaluation windows

Each scenario simulates `estim_days + eval_days` trading days. Moments are estimated on the
first block and portfolios are scored on both blocks, so in-sample and out-of-sample Sharpe
are reported side by side. The submitted code scored on the estimation window only.

### 3.3 Real data

16 large-cap US equities, 2018-2025, daily adjusted closes from Yahoo Finance, with 20
non-overlapping-test walk-forward windows (252-day estimation, 63-day evaluation). Prices
are cached to disk so a rerun is deterministic and needs no network. If `yfinance` is
unavailable the notebook falls back to a clearly-labelled synthetic surrogate so that the
structure still executes, and every affected table is marked accordingly.


In [ ]:
TRADING_DAYS = 252


@dataclass(frozen=True)
class RegimeSpec:
    name: str
    drift: Tuple[float, float]
    vol: Tuple[float, float]
    beta: float          # mean pairwise correlation
    count: int           # scenarios allocated to this regime


REGIMES: Dict[str, RegimeSpec] = {
    "bull":     RegimeSpec("bull",     (0.15, 0.30), (0.10, 0.20), 0.30, 14),
    "bear":     RegimeSpec("bear",     (-0.25, -0.05), (0.25, 0.45), 0.60, 11),
    "choppy":   RegimeSpec("choppy",   (-0.05, 0.05), (0.15, 0.25), 0.15, 6),
    "volatile": RegimeSpec("volatile", (-0.15, 0.15), (0.35, 0.55), 0.45, 7),
    "recovery": RegimeSpec("recovery", (0.05, 0.20), (0.20, 0.35), 0.40, 12),
}


def factor_correlation(n: int, beta: float, rng: np.random.Generator) -> np.ndarray:
    """Single-factor correlation matrix, PSD by construction, mean off-diagonal = beta.

    Loadings are dispersed so that pairwise correlations vary across pairs, then rescaled so
    that the *realised* mean off-diagonal correlation equals the nominal beta rather than
    merely matching it in expectation. Without the rescaling the realised value drifts by a
    percent or two per draw, which is how the submitted study ended up reporting a regime
    parameter it had not simulated.
    """
    beta = float(np.clip(beta, 0.0, 0.95))
    if beta == 0.0:
        return np.eye(n)
    loadings = rng.uniform(0.85, 1.15, n)
    scale = min(math.sqrt(beta) / loadings.mean(), 0.97 / loadings.max())
    a = loadings * scale
    corr = np.outer(a, a)
    np.fill_diagonal(corr, 1.0)
    return corr


def simulate_regime(regime: str, n_assets: int, n_days: int,
                    seed: int) -> Dict[str, object]:
    """Correlated GBM price paths for one regime. Returns prices plus true parameters."""
    spec = REGIMES[regime]
    rng = np.random.default_rng(seed)

    mu_true = rng.uniform(*spec.drift, n_assets)
    vol_true = rng.uniform(*spec.vol, n_assets)
    corr = factor_correlation(n_assets, spec.beta, rng)
    cov_true = corr * np.outer(vol_true, vol_true)

    chol = np.linalg.cholesky(corr + 1e-10 * np.eye(n_assets))
    dt = 1.0 / TRADING_DAYS
    shocks = (chol @ rng.standard_normal((n_assets, n_days - 1))).T
    increments = ((mu_true - 0.5 * vol_true ** 2) * dt
                  + vol_true * np.sqrt(dt) * shocks)

    log_prices = np.vstack([np.zeros(n_assets), np.cumsum(increments, axis=0)])
    prices = 100.0 * np.exp(log_prices)
    dates = pd.bdate_range("2015-01-01", periods=n_days)
    frame = pd.DataFrame(prices, index=dates,
                         columns=[f"A{i:02d}" for i in range(n_assets)])
    return {"prices": frame, "regime": regime, "mu_true": mu_true,
            "cov_true": cov_true, "beta": spec.beta, "seed": seed}


def log_returns(prices: pd.DataFrame) -> pd.DataFrame:
    """Log returns. The submitted code used pct_change, inconsistent with the manuscript."""
    return np.log(prices / prices.shift(1)).dropna(how="all")


def estimate_moments(returns: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """Agent 2a - Market Analyst. Annualised mean vector and covariance matrix."""
    mu = returns.mean().to_numpy() * TRADING_DAYS
    cov = returns.cov().to_numpy() * TRADING_DAYS
    cov = 0.5 * (cov + cov.T)                      # enforce exact symmetry
    eigmin = np.linalg.eigvalsh(cov).min()
    if eigmin < 0:                                  # numerical floor only
        cov += (1e-12 - eigmin) * np.eye(cov.shape[0])
    return mu, cov


def scenario_panel(regime: str, seed: int, cfg: Config = CFG) -> Dict[str, object]:
    """One synthetic scenario split into disjoint estimation and evaluation windows."""
    total = cfg.estim_days + cfg.eval_days
    sim = simulate_regime(regime, cfg.n_assets, total + 1, seed)
    rets = log_returns(sim["prices"])
    estim, evaluation = rets.iloc[:cfg.estim_days], rets.iloc[cfg.estim_days:]
    mu_is, cov_is = estimate_moments(estim)
    mu_oos, cov_oos = estimate_moments(evaluation)
    return {"regime": regime, "seed": seed, "beta": sim["beta"],
            "mu_is": mu_is, "cov_is": cov_is,
            "mu_oos": mu_oos, "cov_oos": cov_oos,
            "realised_beta": float(np.mean(
                estim.corr().to_numpy()[np.triu_indices(cfg.n_assets, k=1)]))}


def regime_schedule(cfg: Config = CFG) -> List[Tuple[str, int]]:
    """Deterministic (regime, seed) schedule with the counts reported in Table 5."""
    total = sum(s.count for s in REGIMES.values())
    scale = cfg.n_scenarios / total
    schedule: List[Tuple[str, int]] = []
    for name, spec in REGIMES.items():
        n = max(1, int(round(spec.count * scale)))
        schedule.extend((name, 1000 + 97 * len(schedule) + k) for k in range(n))
    return schedule[:cfg.n_scenarios] if len(schedule) >= cfg.n_scenarios else schedule


# --- verification of the generator's own claims -------------------------------
_probe = factor_correlation(200, 0.60, np.random.default_rng(0))
_off = _probe[np.triu_indices(200, k=1)]
assert np.linalg.eigvalsh(_probe).min() > -1e-10, "correlation matrix is not PSD"
assert abs(_off.mean() - 0.60) < 0.01, f"mean correlation {_off.mean():.4f} != 0.60"

_schedule = regime_schedule()
_counts = pd.Series([r for r, _ in _schedule]).value_counts()
print(f"PASS  correlation construction is PSD, realised mean correlation "
      f"{_off.mean():.4f} for nominal beta=0.60")
print(f"PASS  regime schedule fixed at {len(_schedule)} scenarios: "
      + ", ".join(f"{k} {v}" for k, v in _counts.items()))

_demo = scenario_panel("bear", 1000)
print(f"\nbear scenario check: nominal beta=0.60, realised in-sample "
      f"correlation={_demo['realised_beta']:.3f}")
print(f"  annualised mean return range "
      f"[{_demo['mu_is'].min():+.2%}, {_demo['mu_is'].max():+.2%}]")


In [ ]:
@dataclass(frozen=True)
class Window:
    index: int
    train_start: int
    train_stop: int      # exclusive; equals test_start so there is no gap and no overlap
    test_start: int
    test_stop: int


def walk_forward_windows(n_obs: int, cfg: Config = CFG) -> List[Window]:
    """Non-overlapping evaluation windows rolled forward by `horizon` observations."""
    windows: List[Window] = []
    span = cfg.lookback + cfg.horizon
    max_windows = 1 + (n_obs - span) // cfg.horizon if n_obs >= span else 0
    for i in range(min(cfg.n_windows, max_windows)):
        train_start = i * cfg.horizon
        train_stop = train_start + cfg.lookback
        windows.append(Window(i, train_start, train_stop, train_stop,
                              train_stop + cfg.horizon))
    return windows


CACHE = OUTDIR / "cache" / "prices.csv"
REAL_DATA_IS_SYNTHETIC = False


def load_real_prices(cfg: Config = CFG) -> pd.DataFrame:
    """Adjusted closes for the 16 tickers, cached so reruns are deterministic."""
    global REAL_DATA_IS_SYNTHETIC
    if CACHE.exists():
        frame = pd.read_csv(CACHE, index_col=0, parse_dates=True)
        REAL_DATA_IS_SYNTHETIC = bool(frame.attrs.get("synthetic", False)) or \
            frame.columns[0].startswith("SURROGATE")
        print(f"loaded cached prices: {frame.shape[0]} rows x {frame.shape[1]} tickers")
        return frame
    try:
        import yfinance as yf
        raw = yf.download(list(cfg.tickers), start=cfg.start, end=cfg.end,
                          auto_adjust=True, progress=False)
        frame = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw
        frame = frame[list(cfg.tickers)].dropna(how="any")
        if frame.empty:
            raise RuntimeError("Yahoo Finance returned no usable rows")
        frame.to_csv(CACHE)
        print(f"downloaded {frame.shape[0]} rows x {frame.shape[1]} tickers "
              f"({frame.index[0].date()} to {frame.index[-1].date()})")
        return frame
    except Exception as exc:                                  # offline / API change
        print(f"NOTE  Yahoo Finance unavailable ({type(exc).__name__}: {exc}).")
        print("      Falling back to a labelled synthetic surrogate. Every table built "
              "from it is flagged, and the real-data numbers must be regenerated with "
              "network access before submission.")
        REAL_DATA_IS_SYNTHETIC = True
        rng = np.random.default_rng(7)
        n_obs = 1900
        vol = rng.uniform(0.18, 0.40, len(cfg.tickers))
        drift = rng.uniform(0.02, 0.22, len(cfg.tickers))
        corr = factor_correlation(len(cfg.tickers), 0.35, rng)
        chol = np.linalg.cholesky(corr + 1e-10 * np.eye(len(cfg.tickers)))
        dt = 1.0 / TRADING_DAYS
        shocks = (chol @ rng.standard_normal((len(cfg.tickers), n_obs - 1))).T
        steps = (drift - 0.5 * vol ** 2) * dt + vol * np.sqrt(dt) * shocks
        prices = 100.0 * np.exp(np.vstack([np.zeros(len(cfg.tickers)),
                                           np.cumsum(steps, axis=0)]))
        frame = pd.DataFrame(prices, index=pd.bdate_range(cfg.start, periods=n_obs),
                             columns=[f"SURROGATE_{t}" for t in cfg.tickers])
        frame.to_csv(CACHE)
        return frame


REAL_PRICES = load_real_prices()
REAL_RETURNS = log_returns(REAL_PRICES)
REAL_WINDOWS = walk_forward_windows(len(REAL_RETURNS))
print(f"walk-forward windows: {len(REAL_WINDOWS)} "
      f"(lookback {CFG.lookback}, horizon {CFG.horizon})")
print(f"real data is synthetic surrogate: {REAL_DATA_IS_SYNTHETIC}")

# Evaluation windows must not overlap each other, and train must abut test.
for a, b in zip(REAL_WINDOWS, REAL_WINDOWS[1:]):
    assert a.test_stop <= b.test_stop and a.train_stop == a.test_start
print("PASS  windows are contiguous in training and non-overlapping in evaluation")


## 4. The corrected QUBO formulation (Editor 2, Comment 1)

The submitted formulation carried the expected-return term twice with opposite signs — once
folded into the diagonal of the quadratic block and once as an explicit linear term. Because
`x_i` is binary, `x_i^2 = x_i`, so the diagonal of the quadratic form *is* a linear term and
the two contributions cancelled. The surrogate the solver actually minimised was therefore
**independent of `mu`**: it was a pure minimum-variance problem wearing a mean-variance label.

### Formulation used here

Maximise expected return net of a variance penalty over the equal-weighted portfolio
implied by `x`, subject to holding exactly `B` assets:

    maximise    (1/B) mu'x  -  (q/B^2) x' Sigma x
    subject to  1'x = B,   x in {0,1}^N

Note the `1/B` and `1/B^2` scalings. The submitted version omitted them, which silently
rescaled the effective risk aversion by a factor of `B` and made the reported `q` not the `q`
in force. Folding the cardinality constraint in as a quadratic penalty gives the QUBO

    minimise  x' Q x  +  c'x  +  const
      Q     = (q/B^2) Sigma  +  lambda * J          (J = 11', the all-ones matrix)
      c     = -(1/B) mu  -  2 * lambda * B * 1
      const = lambda * B^2

The return term now appears **exactly once**, in `c`.

### Choice of the penalty coefficient

Adding or removing one asset moves `(1'x - B)^2` from 0 to at least 1, so any `lambda`
exceeding the largest single-flip change in the unpenalised objective makes every infeasible
state strictly worse than every feasible one. We take 1.5 times that bound, computed per
instance rather than hard-coded, and then *verify* feasibility of the QUBO ground state by
enumeration in the test below.

### Executable proof of correctness

Three properties are asserted, not asserted-in-prose:

1. `qubo_energy(x)` equals the direct objective for every feasible `x` (up to a constant).
2. The QUBO ground state satisfies `1'x = B` — the penalty is genuinely sufficient.
3. **Regression test for the original bug:** replacing `mu` with a materially different
   vector changes the QUBO's optimal selection. Under the submitted formulation this test
   fails, because the optimum did not depend on `mu` at all.


In [ ]:
@dataclass
class QUBO:
    Q: np.ndarray
    c: np.ndarray
    const: float
    lam: float
    B: int
    q_risk: float


def default_lambda(mu: np.ndarray, cov: np.ndarray, b: int, q_risk: float) -> float:
    """Smallest single-flip-safe penalty, times a 1.5 safety factor."""
    linear = np.abs(mu) / b
    quad = 2.0 * q_risk * np.abs(cov).sum(axis=1) / (b ** 2)
    return 1.5 * float(np.max(linear + quad)) + 1e-9


def build_qubo(mu: np.ndarray, cov: np.ndarray, b: int, q_risk: float,
               lam: Optional[float] = None) -> QUBO:
    n = mu.size
    lam = default_lambda(mu, cov, b, q_risk) if lam is None else lam
    Q = (q_risk / b ** 2) * cov + lam * np.ones((n, n))
    c = -(mu / b) - 2.0 * lam * b * np.ones(n)
    return QUBO(Q=Q, c=c, const=lam * b ** 2, lam=lam, B=b, q_risk=q_risk)


def qubo_energy(qubo: QUBO, x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    return float(x @ qubo.Q @ x + qubo.c @ x + qubo.const)


def objective_direct(mu: np.ndarray, cov: np.ndarray, b: int, q_risk: float,
                     x: np.ndarray) -> float:
    """The mean-variance objective as a minimisation, ignoring the penalty."""
    x = np.asarray(x, dtype=float)
    k = x.sum()
    if k == 0:
        return 0.0
    w = x / b                      # equal weights implied by the budget, not by |S|
    return float(q_risk * (w @ cov @ w) - (mu @ w))


def bit_table(n: int) -> np.ndarray:
    """All 2^n binary vectors, as uint8. Only used for verification and enumeration."""
    idx = np.arange(2 ** n, dtype=np.uint64)
    return ((idx[:, None] >> np.arange(n, dtype=np.uint64)) & 1).astype(np.uint8)


def qubo_energies_all(qubo: QUBO, n: int, chunk: int = 1 << 16) -> np.ndarray:
    """Diagonal of the QUBO cost operator over the full computational basis.

    Chunked so that N up to ~22 qubits stays inside a few hundred MB.
    """
    total = 2 ** n
    out = np.empty(total, dtype=np.float64)
    powers = np.arange(n, dtype=np.uint64)
    for start in range(0, total, chunk):
        stop = min(start + chunk, total)
        idx = np.arange(start, stop, dtype=np.uint64)
        bits = ((idx[:, None] >> powers) & 1).astype(np.float64)
        out[start:stop] = (np.einsum("ij,jk,ik->i", bits, qubo.Q, bits)
                           + bits @ qubo.c + qubo.const)
    return out


# --- executable proof ---------------------------------------------------------
_p = scenario_panel("bull", 1000)
_mu, _cov = _p["mu_is"], _p["cov_is"]
_n, _b = CFG.n_assets, CFG.budget
_qubo = build_qubo(_mu, _cov, _b, CFG.q_risk)

# (1) On feasible states the QUBO reproduces the direct objective up to a constant.
_offsets = []
for _combo in itertools.islice(itertools.combinations(range(_n), _b), 400):
    _x = np.zeros(_n)
    _x[list(_combo)] = 1.0
    _offsets.append(qubo_energy(_qubo, _x) - objective_direct(_mu, _cov, _b, CFG.q_risk, _x))
_spread = float(np.ptp(_offsets))
assert _spread < 1e-9, f"QUBO and direct objective differ by a non-constant: {_spread:.3e}"
print(f"PASS  (1) QUBO matches the direct objective on all feasible states "
      f"(offset spread {_spread:.2e})")

# (2) The unconstrained QUBO ground state is feasible, so lambda is sufficient.
_energies = qubo_energies_all(_qubo, _n)
_ground = int(np.argmin(_energies))
_ground_bits = bit_table(_n)[_ground]
assert _ground_bits.sum() == _b, (
    f"QUBO ground state holds {_ground_bits.sum()} assets, expected {_b}; "
    f"lambda={_qubo.lam:.4f} is too small")
print(f"PASS  (2) QUBO ground state is feasible: holds {int(_ground_bits.sum())} assets "
      f"with lambda={_qubo.lam:.4f}")

# (3) Regression test for the cancellation bug: the optimum must depend on mu.
_mu_alt = _mu[::-1] * 3.0 - 0.5
_sel_a = tuple(np.flatnonzero(_ground_bits))
_sel_b = tuple(np.flatnonzero(
    bit_table(_n)[int(np.argmin(qubo_energies_all(
        build_qubo(_mu_alt, _cov, _b, CFG.q_risk), _n)))]))
assert _sel_a != _sel_b, (
    "REGRESSION: the QUBO optimum is invariant to mu. The expected-return term is "
    "cancelling against the quadratic diagonal - this is the submitted formulation's bug.")
print(f"PASS  (3) optimum responds to mu: {_sel_a} -> {_sel_b}")

del _energies


## 5. Tier 2 — QAOA, fully specified (Editor 1, Comment 3)

The submitted code contained **no QAOA**. It called `NumPyMinimumEigensolver`, which
diagonalises the Hamiltonian exactly and returns the true ground state. `QAOA`, `COBYLA` and
`Sampler` were imported and never referenced. Consequently the manuscript's Section 3.4 —
depth, shot count, optimiser, initial state — described an algorithm that was never run, and
the "quantum" results were exact classical optima. That is the single most serious defect in
the submission and it is what Editor 1's Comment 3 was probing.

What follows is a genuine QAOA. Every element the reviewer asked to see named:

| Element | Value |
|---|---|
| Encoding | one binary variable per asset, `x_i = 1` iff asset `i` is held; `N` qubits |
| Cost Hamiltonian | `H_C` diagonal in the computational basis, `<z|H_C|z> = ` QUBO energy of `z` (Section 4) |
| Mixer | transverse field `H_M = sum_j X_j`, realised as `RX(2*beta)` on every qubit |
| Initial state | uniform superposition `H^{tensor N}|0>`, the ground state of `-H_M` |
| Depth | `p = 2`, so four variational parameters `(gamma_1, beta_1, gamma_2, beta_2)` |
| Outer optimiser | COBYLA, `maxiter = 150`, 3 random restarts, best restart retained |
| Objective | `<psi(gamma, beta)|H_C|psi(gamma, beta)>`, evaluated exactly from the state vector |
| Measurement | 1024 shots drawn from the exact Born distribution `|amplitude|^2` |
| Simulator | exact state-vector simulation in NumPy, `complex128`; cross-validated against Qiskit Aer in Section 6.1 |
| Simulable ceiling | 22 qubits (state vector `2^22` amplitudes); larger instances are reported as not simulable rather than silently substituted |

**Conditioning note.** Cost energies are affinely rescaled to zero mean and unit variance
before entering the phase separator. Since `<(H_C - m)/s> = (<H_C> - m)/s`, this leaves the
optimal parameters' *argmin* unchanged while keeping `gamma` in a well-scaled range; without
it COBYLA stalls because raw energies are O(lambda*B^2). The reported energies and
approximation ratios always use the unscaled Hamiltonian.

## 6. Converting measurements into a portfolio

QAOA returns a distribution over `2^N` bitstrings, most of which violate `1'x = B`. The
submitted code had no defined rule for this step. The rule used here is explicit and
deterministic:

1. Keep the measured bitstrings with exactly `B` ones. These are feasible.
2. Score each *distinct* feasible bitstring by the true objective `f` and take the best.
3. If no measured bitstring is feasible, repair the most frequently measured one by greedily
   adding or removing the asset with the best marginal `f` until it holds exactly `B`.
4. The winner becomes the Tier-3 seed.

The feasible fraction in step 1 is recorded per instance and reported, because it is the
honest measure of how much of the quantum layer's output is usable.


In [ ]:
@dataclass
class QAOAResult:
    params: np.ndarray
    energy_expectation: float      # <H_C> at the optimal parameters, unscaled
    counts: Dict[int, int]         # basis-state index -> shot count
    feasible_fraction: float
    n_surrogate_evals: int
    ground_energy: Optional[float] = None
    approx_ratio: Optional[float] = None
    p_optimal: Optional[float] = None
    seconds: float = 0.0
    simulable: bool = True


class CostOperator:
    """Diagonal cost Hamiltonian for a QUBO, precomputed over the computational basis."""

    def __init__(self, qubo: QUBO, n: int) -> None:
        self.n = n
        self.energies = qubo_energies_all(qubo, n)
        self.popcount = np.bitwise_count(np.arange(2 ** n, dtype=np.uint32)) \
            if hasattr(np, "bitwise_count") else \
            np.array([bin(i).count("1") for i in range(2 ** n)], dtype=np.uint8)
        mean, std = self.energies.mean(), self.energies.std()
        self._scaled = (self.energies - mean) / (std if std > 0 else 1.0)

    @property
    def scaled(self) -> np.ndarray:
        return self._scaled


def _apply_mixer(psi: np.ndarray, beta: float, n: int) -> np.ndarray:
    """exp(-i*beta*sum_j X_j) applied qubit by qubit, i.e. RX(2*beta) on each wire."""
    cos_b, sin_b = math.cos(beta), math.sin(beta)
    for j in range(n):
        block = psi.reshape(2 ** (n - j - 1), 2, 2 ** j)
        lo, hi = block[:, 0, :].copy(), block[:, 1, :].copy()
        block[:, 0, :] = cos_b * lo - 1j * sin_b * hi
        block[:, 1, :] = -1j * sin_b * lo + cos_b * hi
        psi = block.reshape(-1)
    return psi


def qaoa_statevector(cost: CostOperator, params: np.ndarray) -> np.ndarray:
    """|psi(gamma, beta)> for depth p = len(params)//2, starting from uniform superposition."""
    n = cost.n
    dim = 2 ** n
    psi = np.full(dim, 1.0 / math.sqrt(dim), dtype=np.complex128)
    scaled = cost.scaled
    for layer in range(len(params) // 2):
        gamma, beta = params[2 * layer], params[2 * layer + 1]
        psi *= np.exp(-1j * gamma * scaled)
        psi = _apply_mixer(psi, float(beta), n)
    return psi


def run_qaoa(qubo: QUBO, n: int, cfg: QAOAConfig,
             ground_energy: Optional[float] = None) -> QAOAResult:
    """Optimise the variational parameters, then sample `cfg.shots` measurements.

    `ground_energy` is the reference for the approximation ratio. When omitted it is taken
    as the minimum energy over the *feasible* subspace (exactly `B` assets held), which is
    the only comparison that means anything for a constrained problem. It must be recomputed
    per surrogate, since each risk aversion in the Tier-1 spectrum defines a different
    Hamiltonian.
    """
    if n > cfg.max_qubits:
        return QAOAResult(params=np.array([]), energy_expectation=float("nan"),
                          counts={}, feasible_fraction=float("nan"),
                          n_surrogate_evals=0, simulable=False)

    started = time.perf_counter()
    cost = CostOperator(qubo, n)
    scaled = cost.scaled
    n_evals = 0
    if ground_energy is None:
        feasible_states = cost.popcount == qubo.B
        ground_energy = float(cost.energies[feasible_states].min())

    def expectation(params: np.ndarray) -> float:
        nonlocal n_evals
        n_evals += 1
        probs = np.abs(qaoa_statevector(cost, params)) ** 2
        return float(probs @ scaled)

    rng = np.random.default_rng(cfg.seed)
    best_params, best_value = None, np.inf
    for _ in range(cfg.n_restarts):
        x0 = np.concatenate([rng.uniform(0, 2 * np.pi, cfg.depth),
                             rng.uniform(0, np.pi, cfg.depth)])
        x0 = x0.reshape(2, cfg.depth).T.reshape(-1)          # interleave gamma, beta
        out = minimize(expectation, x0, method=cfg.optimizer,
                       options={"maxiter": cfg.maxiter, "rhobeg": 0.35})
        if out.fun < best_value:
            best_value, best_params = float(out.fun), np.asarray(out.x, dtype=float)

    psi = qaoa_statevector(cost, best_params)
    probs = np.abs(psi) ** 2
    probs = probs / probs.sum()

    shot_idx = rng.choice(probs.size, size=cfg.shots, p=probs)
    values, counts_arr = np.unique(shot_idx, return_counts=True)
    counts = {int(v): int(c) for v, c in zip(values, counts_arr)}

    feasible_mask = cost.popcount[values] == qubo.B
    feasible_fraction = float(counts_arr[feasible_mask].sum() / cfg.shots)

    energy_expectation = float(probs @ cost.energies)

    # Normalise the approximation ratio inside the feasible subspace. Using the global
    # maximum would include heavily penalised infeasible states and inflate the ratio
    # towards 1 regardless of how well the ansatz performed.
    feasible_energies = cost.energies[cost.popcount == qubo.B]
    worst = float(feasible_energies.max())
    denom = worst - ground_energy
    approx_ratio = float((worst - energy_expectation) / denom) if denom > 0 else 1.0
    optimal_states = np.flatnonzero(
        np.isclose(cost.energies, ground_energy, rtol=0, atol=1e-9))
    p_optimal = float(probs[optimal_states].sum())

    return QAOAResult(params=best_params, energy_expectation=energy_expectation,
                      counts=counts, feasible_fraction=feasible_fraction,
                      n_surrogate_evals=n_evals, ground_energy=ground_energy,
                      approx_ratio=approx_ratio, p_optimal=p_optimal,
                      seconds=time.perf_counter() - started)


def bits_of(index: int, n: int) -> np.ndarray:
    return np.array([(index >> j) & 1 for j in range(n)], dtype=np.uint8)


def repair_to_budget(selection: List[int], b: int, n: int,
                     score: Callable[[Sequence[int]], float]) -> List[int]:
    """Greedily add or drop assets by marginal objective until exactly b are held."""
    current = sorted(set(selection))
    while len(current) > b:
        current = min((([a for a in current if a != drop]) for drop in current),
                      key=lambda cand: -score(cand))
    while len(current) < b:
        outside = [a for a in range(n) if a not in current]
        current = sorted(max((current + [add] for add in outside),
                             key=lambda cand: score(cand)))
    return sorted(current)


def select_portfolio(result: QAOAResult, n: int, b: int,
                     score: Callable[[Sequence[int]], float]) -> Tuple[List[int], str]:
    """The four-step measurement-to-portfolio rule documented in Section 6."""
    if not result.counts:
        return [], "no-measurements"

    feasible = [idx for idx in result.counts
                if int(bits_of(idx, n).sum()) == b]
    if feasible:
        best = max(feasible, key=lambda idx: score(np.flatnonzero(bits_of(idx, n))))
        return sorted(int(i) for i in np.flatnonzero(bits_of(best, n))), "feasible-sample"

    most_frequent = max(result.counts, key=result.counts.get)
    raw = [int(i) for i in np.flatnonzero(bits_of(most_frequent, n))]
    return repair_to_budget(raw, b, n, score), "repaired"


print("QAOA simulator ready.")


### 6.1 Validating the simulator

Three independent checks, because a hand-written state-vector simulator is exactly the kind
of component a reviewer is entitled to distrust:

1. **Unitarity** — the mixer preserves the norm of the state to machine precision.
2. **Dense-matrix cross-check** — for a 6-qubit instance the state is recomputed from
   explicit Kronecker products of `RX(2*beta)` matrices and a dense diagonal phase operator,
   a completely separate code path from the in-place reshape used in production. The two
   state vectors must agree to `1e-12`.
3. **Qiskit Aer cross-check** — if Qiskit is installed, the same circuit is built with
   `QuantumCircuit` and evolved by Aer's state-vector backend. Agreement here confirms
   qubit ordering and gate conventions against a reference implementation, and gives the
   manuscript a citable simulator name.

A depth sweep is also reported: the approximation ratio must increase with `p`, which is the
qualitative signature of QAOA working as intended.


In [ ]:
from functools import reduce

QISKIT_INFO = "not installed"

# --- check 1: unitarity -------------------------------------------------------
_n_small = 6
_psi = np.full(2 ** _n_small, 1.0 / math.sqrt(2 ** _n_small), dtype=np.complex128)
_out = _apply_mixer(_psi.copy(), 0.7, _n_small)
assert abs(np.linalg.norm(_out) - 1.0) < 1e-12, "mixer is not unitary"
print(f"PASS  (1) mixer preserves norm to {abs(np.linalg.norm(_out) - 1.0):.1e}")

# --- check 2: dense Kronecker cross-check -------------------------------------
_p_small = scenario_panel("choppy", 2000)
_mu6, _cov6 = _p_small["mu_is"][:_n_small], _p_small["cov_is"][:_n_small, :_n_small]
_qubo6 = build_qubo(_mu6, _cov6, 3, CFG.q_risk)
_cost6 = CostOperator(_qubo6, _n_small)
_params = np.array([0.83, 0.41, 1.27, 0.62])

def _dense_statevector(cost: CostOperator, params: np.ndarray) -> np.ndarray:
    dim = 2 ** cost.n
    psi = np.full(dim, 1.0 / math.sqrt(dim), dtype=np.complex128)
    for layer in range(len(params) // 2):
        gamma, beta = params[2 * layer], params[2 * layer + 1]
        psi = np.diag(np.exp(-1j * gamma * cost.scaled)) @ psi
        rx = np.array([[math.cos(beta), -1j * math.sin(beta)],
                       [-1j * math.sin(beta), math.cos(beta)]], dtype=np.complex128)
        psi = reduce(np.kron, [rx] * cost.n) @ psi
    return psi

_delta = np.abs(qaoa_statevector(_cost6, _params) - _dense_statevector(_cost6, _params)).max()
assert _delta < 1e-12, f"reshape and dense simulators disagree by {_delta:.3e}"
print(f"PASS  (2) reshape simulator matches dense Kronecker reference to {_delta:.1e}")

# --- check 3: Qiskit Aer cross-check (optional) --------------------------------
try:
    import qiskit
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector

    qc = QuantumCircuit(_n_small)
    qc.h(range(_n_small))
    for _layer in range(len(_params) // 2):
        _g, _b = _params[2 * _layer], _params[2 * _layer + 1]
        qc.diagonal(list(np.exp(-1j * _g * _cost6.scaled)), list(range(_n_small)))
        for _j in range(_n_small):
            qc.rx(2 * _b, _j)
    _qk = np.asarray(Statevector.from_instruction(qc).data)
    _delta_qk = np.abs(_qk - qaoa_statevector(_cost6, _params)).max()
    QISKIT_INFO = f"qiskit {qiskit.__version__}"
    assert _delta_qk < 1e-10, f"Qiskit disagrees by {_delta_qk:.3e}"
    print(f"PASS  (3) matches {QISKIT_INFO} Statevector to {_delta_qk:.1e}")
except ImportError:
    print("SKIP  (3) Qiskit not installed; checks 1 and 2 already pin the simulator. "
          "Install qiskit to enable the reference cross-check.")
except Exception as exc:
    print(f"SKIP  (3) Qiskit cross-check unavailable ({type(exc).__name__}: {exc})")

# --- depth sweep: approximation ratio should improve with p --------------------
_ground6 = float(_cost6.energies[_cost6.popcount == 3].min())
print("\nDepth sweep on the 6-qubit instance (N=6, B=3):")
print(f"{'p':>3} {'<H_C>':>12} {'approx ratio':>13} {'P(optimal)':>11} "
      f"{'feasible %':>11} {'sec':>6}")
_sweep = {}
for _p in (1, 2, 3, 4):
    _cfg_p = QAOAConfig(depth=_p, shots=CFG.qaoa.shots, maxiter=200, n_restarts=4,
                        seed=CFG.qaoa.seed)
    _r = run_qaoa(_qubo6, _n_small, _cfg_p, ground_energy=_ground6)
    _sweep[_p] = _r
    print(f"{_p:>3} {_r.energy_expectation:>12.4f} {_r.approx_ratio:>13.4f} "
          f"{_r.p_optimal:>11.4f} {100 * _r.feasible_fraction:>10.1f}% {_r.seconds:>6.2f}")

assert _sweep[4].approx_ratio > _sweep[1].approx_ratio, \
    "approximation ratio did not improve with depth - the ansatz is not being optimised"
print("\nPASS  approximation ratio improves monotonically enough with depth "
      f"({_sweep[1].approx_ratio:.4f} at p=1 -> {_sweep[4].approx_ratio:.4f} at p=4)")


## 7. Tier 3 — classical local refinement

Steepest-ascent pairwise swap on the true objective `f`, starting from the Tier-2 seed. One
sweep evaluates every (held, not-held) exchange and commits the single best improvement;
sweeps continue until no exchange gains more than `eps`. This is the tier that bridges the
quadratic surrogate and the actual ratio objective.

Unlike the submitted code, refinement is **not** allowed to consult a baseline solution, and
it draws from the same `EvalCounter`, so its cost is visible in the reported evaluation
totals rather than hidden.

## 8. Baselines and the exact optimum (Editor 2, Comment 5)

The submission compared against two baselines, one of which was contained inside the
proposed method. The full set is now:

| Baseline | Type | Cost accounting |
|---|---|---|
| Exhaustive enumeration | exact | `C(N,B)` evaluations; the ground truth |
| Classical MVO | deterministic, continuous relaxation then top-`B` truncation | one convex solve |
| Strong greedy | deterministic forward selection by marginal `f` | `~B*N` evaluations |
| Random sampling | stochastic | full budget |
| Genetic algorithm | stochastic, random-key encoding | full budget |
| Particle swarm | stochastic, random-key encoding | full budget |
| Differential evolution | stochastic, random-key encoding | full budget |
| Simulated annealing | stochastic, swap neighbourhood | full budget |

**Enumeration is the decisive addition.** At `N=16, B=5` there are only 4368 feasible
portfolios, so the exact optimum is computable in milliseconds. Every method can therefore be
scored by *optimality gap* rather than by pairwise win rate. A win rate cannot distinguish
"my method is better" from "both methods found the same optimum and ties were broken by
floating-point noise" — which is what was happening in the submission. The gap can.

The three random-key methods (GA, PSO, DE) encode a portfolio as a vector in `[0,1]^N`
decoded by taking its `B` largest components. This keeps every candidate feasible by
construction, so no method is advantaged by a repair heuristic the others lack.


In [ ]:
ENUM_LIMIT = 2_000_000     # beyond this, exhaustive enumeration is declared intractable


def local_refine(counter: EvalCounter, selection: Sequence[int], n: int,
                 eps: float = 1e-10) -> List[int]:
    """Tier 3: steepest-ascent pairwise swaps until no exchange improves f by > eps."""
    current = sorted(selection)
    if not current:
        return current
    try:
        best_value = counter(current)
        improving = True
        while improving:
            improving = False
            challenger, challenger_value = current, best_value
            for held in current:
                for candidate in range(n):
                    if candidate in current:
                        continue
                    trial = sorted([a for a in current if a != held] + [candidate])
                    value = counter(trial)
                    if value > challenger_value + eps:
                        challenger, challenger_value = trial, value
            if challenger_value > best_value + eps:
                current, best_value, improving = challenger, challenger_value, True
    except BudgetExhausted:
        pass
    return sorted(counter.best_selection) if counter.best_selection else current


def enumerate_optimum(mu: np.ndarray, cov: np.ndarray, b: int,
                      chunk: int = 200_000) -> Optional[Dict[str, object]]:
    """Exact optimum by exhaustive enumeration, or None if C(N,B) exceeds ENUM_LIMIT."""
    n = mu.size
    total = math.comb(n, b)
    if total > ENUM_LIMIT:
        return None
    started = time.perf_counter()
    best_value, best_combo = -np.inf, None
    stream = itertools.combinations(range(n), b)
    while True:
        block = list(itertools.islice(stream, chunk))
        if not block:
            break
        combos = np.asarray(block, dtype=np.int64)
        rets = mu[combos].sum(axis=1) / b
        blocks = cov[combos[:, :, None], combos[:, None, :]]
        vols = np.sqrt(np.maximum(blocks.sum(axis=(1, 2)), 0.0)) / b
        with np.errstate(divide="ignore", invalid="ignore"):
            values = np.where(vols > 1e-12, rets / vols, -np.inf)
        winner = int(np.argmax(values))
        if values[winner] > best_value:
            best_value, best_combo = float(values[winner]), combos[winner].tolist()
    return {"selection": sorted(best_combo), "sharpe": best_value,
            "n_evals": total, "seconds": time.perf_counter() - started}


# --- verification: enumeration agrees with the QUBO ground state ---------------
# The QUBO surrogate and the Sharpe objective are different functions, so their optima need
# not coincide; what must hold is that enumeration finds the true Sharpe optimum. We confirm
# that by checking no feasible portfolio beats it.
_exact = enumerate_optimum(_mu, _cov, CFG.budget)
_all_feasible = [sharpe(_mu, _cov, c)
                 for c in itertools.combinations(range(CFG.n_assets), CFG.budget)]
assert abs(_exact["sharpe"] - max(_all_feasible)) < 1e-12
print(f"PASS  exhaustive enumeration is exact: Sharpe {_exact['sharpe']:.6f} over "
      f"{_exact['n_evals']:,} portfolios in {_exact['seconds'] * 1000:.1f} ms")
print(f"      optimal selection: {_exact['selection']}")
print(f"      QUBO surrogate ground state: {list(_sel_a)}  "
      f"(surrogate and objective differ - this is what Tier 3 exists to close)")


In [ ]:
def _finish(counter: EvalCounter, fallback: Sequence[int]) -> List[int]:
    return sorted(counter.best_selection) if counter.best_selection else sorted(fallback)


def classical_mvo(counter: EvalCounter, b: int, seed: int = 0) -> List[int]:
    """Long-only continuous mean-variance solve, then truncate to the top-B weights."""
    mu, cov = counter.mu, counter.cov
    n = mu.size

    def neg_utility(w: np.ndarray) -> float:
        return float(CFG.q_risk * (w @ cov @ w) - mu @ w)

    out = minimize(neg_utility, np.full(n, 1.0 / n), method="SLSQP",
                   bounds=[(0.0, 1.0)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0}],
                   options={"maxiter": 300, "ftol": 1e-12})
    weights = out.x if out.x is not None else np.full(n, 1.0 / n)
    selection = top_b(np.asarray(weights), b)
    counter(selection)
    return _finish(counter, selection)


def strong_greedy(counter: EvalCounter, b: int, seed: int = 0) -> List[int]:
    """Forward selection: repeatedly add the asset with the best marginal objective."""
    n = counter.mu.size
    current: List[int] = []
    try:
        for _ in range(b):
            best_add, best_value = None, -np.inf
            for candidate in range(n):
                if candidate in current:
                    continue
                value = counter(current + [candidate])
                if value > best_value:
                    best_add, best_value = candidate, value
            current = sorted(current + [best_add])
    except BudgetExhausted:
        pass
    return _finish(counter, current)


def random_sampling(counter: EvalCounter, b: int, seed: int = 0) -> List[int]:
    rng = np.random.default_rng(seed)
    n = counter.mu.size
    try:
        while counter.remaining:
            counter(sorted(rng.choice(n, size=b, replace=False).tolist()))
    except BudgetExhausted:
        pass
    return _finish(counter, [])


def _decode(vector: np.ndarray, b: int) -> List[int]:
    """Random-key decoding: hold the b assets with the largest keys."""
    return top_b(vector, b)


def genetic_algorithm(counter: EvalCounter, b: int, seed: int = 0,
                      pop_size: int = 40, mutation: float = 0.15) -> List[int]:
    rng = np.random.default_rng(seed)
    n = counter.mu.size
    try:
        pop = rng.random((pop_size, n))
        fitness = np.array([counter(_decode(ind, b)) for ind in pop])
        while counter.remaining:
            order = np.argsort(-fitness)
            pop, fitness = pop[order], fitness[order]
            elite = max(2, pop_size // 5)

            def tournament() -> np.ndarray:
                # pop is sorted best-first, so the smaller index is the fitter individual.
                contenders = rng.integers(0, pop_size, 2)
                return pop[int(contenders.min())]

            children = []
            while len(children) < pop_size - elite:
                alpha = rng.random(n)
                child = alpha * tournament() + (1 - alpha) * tournament()
                mask = rng.random(n) < mutation
                child[mask] = rng.random(int(mask.sum()))
                children.append(np.clip(child, 0.0, 1.0))
            pop = np.vstack([pop[:elite], np.asarray(children)])
            fitness = np.concatenate([
                fitness[:elite],
                np.array([counter(_decode(ind, b)) for ind in pop[elite:]])])
    except BudgetExhausted:
        pass
    return _finish(counter, [])


def particle_swarm(counter: EvalCounter, b: int, seed: int = 0, n_particles: int = 30,
                   inertia: float = 0.7, c1: float = 1.5, c2: float = 1.5) -> List[int]:
    rng = np.random.default_rng(seed)
    n = counter.mu.size
    try:
        pos = rng.random((n_particles, n))
        vel = rng.uniform(-0.1, 0.1, (n_particles, n))
        personal_best = pos.copy()
        personal_value = np.array([counter(_decode(p, b)) for p in pos])
        g_idx = int(np.argmax(personal_value))
        global_best = personal_best[g_idx].copy()
        while counter.remaining:
            r1, r2 = rng.random((n_particles, n)), rng.random((n_particles, n))
            vel = (inertia * vel + c1 * r1 * (personal_best - pos)
                   + c2 * r2 * (global_best - pos))
            pos = np.clip(pos + vel, 0.0, 1.0)
            values = np.array([counter(_decode(p, b)) for p in pos])
            better = values > personal_value
            personal_best[better], personal_value[better] = pos[better], values[better]
            g_idx = int(np.argmax(personal_value))
            global_best = personal_best[g_idx].copy()
    except BudgetExhausted:
        pass
    return _finish(counter, [])


def differential_evolution(counter: EvalCounter, b: int, seed: int = 0,
                           pop_size: int = 30, F: float = 0.6,
                           CR: float = 0.9) -> List[int]:
    rng = np.random.default_rng(seed)
    n = counter.mu.size
    try:
        pop = rng.random((pop_size, n))
        fitness = np.array([counter(_decode(ind, b)) for ind in pop])
        while counter.remaining:
            for i in range(pop_size):
                a, c, d = rng.choice([j for j in range(pop_size) if j != i],
                                     size=3, replace=False)
                mutant = np.clip(pop[a] + F * (pop[c] - pop[d]), 0.0, 1.0)
                cross = rng.random(n) < CR
                cross[rng.integers(0, n)] = True
                trial = np.where(cross, mutant, pop[i])
                value = counter(_decode(trial, b))
                if value > fitness[i]:
                    pop[i], fitness[i] = trial, value
    except BudgetExhausted:
        pass
    return _finish(counter, [])


def simulated_annealing(counter: EvalCounter, b: int, seed: int = 0,
                        t0: float = 0.35, t_end: float = 1e-3) -> List[int]:
    """Direct swap neighbourhood with geometric cooling over the whole budget."""
    rng = np.random.default_rng(seed)
    n = counter.mu.size
    current = sorted(rng.choice(n, size=b, replace=False).tolist())
    try:
        value = counter(current)
        steps = max(1, counter.budget - 1)
        decay = (t_end / t0) ** (1.0 / steps)
        temperature = t0
        while counter.remaining:
            trial = current.copy()
            trial[rng.integers(0, b)] = int(rng.choice(
                [a for a in range(n) if a not in current]))
            trial = sorted(trial)
            trial_value = counter(trial)
            if (trial_value > value
                    or rng.random() < math.exp((trial_value - value) / max(temperature, 1e-12))):
                current, value = trial, trial_value
            temperature *= decay
    except BudgetExhausted:
        pass
    return _finish(counter, current)


BASELINES: Dict[str, Callable] = {
    "Classical MVO": classical_mvo,
    "Strong Greedy": strong_greedy,
    "Random Sampling": random_sampling,
    "Genetic Algorithm": genetic_algorithm,
    "Particle Swarm": particle_swarm,
    "Differential Evolution": differential_evolution,
    "Simulated Annealing": simulated_annealing,
}

# --- smoke check: every baseline returns a feasible portfolio -------------------
print(f"{'baseline':<24}{'Sharpe':>10}{'gap %':>9}{'f-evals':>10}")
for _name, _fn in BASELINES.items():
    _ctr = EvalCounter(_mu, _cov, CFG.eval_budget)
    _sel = _fn(_ctr, CFG.budget, seed=1)
    assert len(_sel) == CFG.budget, f"{_name} returned {len(_sel)} assets"
    _gap = 100 * (_exact["sharpe"] - _ctr.best_value) / abs(_exact["sharpe"])
    print(f"{_name:<24}{_ctr.best_value:>10.4f}{_gap:>9.3f}{_ctr.n_evals:>10,}")
print(f"\nexact optimum{'':<11}{_exact['sharpe']:>10.4f}{0.0:>9.3f}"
      f"{_exact['n_evals']:>10,}")


## 9. The three-tier pipeline, made fair

The architecture the manuscript describes is preserved:

- **Tier 1 — multi-risk spectrum.** Build a QUBO surrogate at each risk aversion in
  `q in {0.25, 0.5, 1.0}`. Different `q` values tilt the surrogate's optimum along the
  efficient frontier, giving Tier 2 several distinct regions to sample from.
- **Tier 2 — QAOA global sampling.** Run QAOA on each surrogate and convert measurements to
  a candidate portfolio using the Section 6 rule. Keep the best candidate under `f`.
- **Tier 3 — classical local refinement.** Steepest-ascent swaps from that seed.

Three things are different from the submitted implementation, and they are the difference
between a comparison and a foregone conclusion:

1. **No baseline seeding.** Tier 2 never sees the greedy solution. In the submitted code
   `best_assets` was initialised to `greedy_assets` and returned unchanged if nothing beat
   it, which guaranteed `proposed >= greedy` on every scenario by construction.
2. **No target chasing.** The submitted `if best_sharpe < greedy_sharpe * 1.01:` branch,
   which kept searching until the proposed method beat greedy by 1%, is gone.
3. **Shared budget.** All `f` evaluations across all three tiers are drawn from the same
   `EvalCounter` as the baselines receive, so `n_evals` is directly comparable.

`tier2` is a switch (`qaoa`, `random`, `greedy`, `sa`, `none`) so that Section 11 can replace
the quantum layer with a classical stand-in of matched cost and measure what it was actually
contributing. That substitution is the experiment Editor 1's Comment 2 asks for.


In [ ]:
@dataclass
class PipelineConfig:
    budget: int = CFG.budget
    tier1_spectrum: Tuple[float, ...] = (0.25, 0.5, 1.0)
    tier2: str = "qaoa"                 # qaoa | random | greedy | sa | none
    tier3: bool = True
    eval_budget: int = CFG.eval_budget
    qaoa: QAOAConfig = field(default_factory=lambda: CFG.qaoa)
    seed: int = 0


@dataclass
class PipelineResult:
    selection: List[int]
    sharpe: float
    n_evals: int
    seconds: float
    seed_source: str
    simulable: bool = True
    qaoa_seconds: float = 0.0
    qaoa_surrogate_evals: int = 0
    feasible_fraction: float = float("nan")
    approx_ratio: float = float("nan")
    p_optimal: float = float("nan")
    trajectory: List[Tuple[int, float]] = field(default_factory=list)


def _tier2_candidates(counter: EvalCounter, cfg: PipelineConfig,
                      n: int) -> Tuple[List[List[int]], Dict]:
    """Produce candidate seeds using the configured Tier-2 mechanism."""
    b = cfg.budget
    spectrum = cfg.tier1_spectrum
    diag = {"qaoa_seconds": 0.0, "surrogate_evals": 0,
            "feasible": [], "approx": [], "p_opt": [], "simulable": True}
    candidates: List[List[int]] = []
    score = lambda sel: sharpe(counter.mu, counter.cov, sel, counter.rf)

    if cfg.tier2 == "qaoa":
        for q_risk in spectrum:
            qubo = build_qubo(counter.mu, counter.cov, b, q_risk)
            result = run_qaoa(qubo, n, cfg.qaoa)
            if not result.simulable:
                diag["simulable"] = False
                return [], diag
            diag["qaoa_seconds"] += result.seconds
            diag["surrogate_evals"] += result.n_surrogate_evals
            diag["feasible"].append(result.feasible_fraction)
            if result.approx_ratio is not None:
                diag["approx"].append(result.approx_ratio)
                diag["p_opt"].append(result.p_optimal)
            selection, _ = select_portfolio(result, n, b, score)
            if selection:
                candidates.append(selection)

    elif cfg.tier2 == "random":
        rng = np.random.default_rng(cfg.seed + 11)
        candidates = [sorted(rng.choice(n, size=b, replace=False).tolist())
                      for _ in spectrum]

    elif cfg.tier2 == "greedy":
        # One greedy seed per risk level, tilting mu by the surrogate's variance penalty.
        for q_risk in spectrum:
            tilted = counter.mu - q_risk * np.diag(counter.cov) / b
            candidates.append(top_b(tilted, b))

    elif cfg.tier2 == "sa":
        # Cost-matched classical stand-in: a short SA run per risk level, given the same
        # number of surrogate evaluations QAOA's outer loop consumes.
        matched = cfg.qaoa.maxiter * cfg.qaoa.n_restarts
        for k, _ in enumerate(spectrum):
            probe = EvalCounter(counter.mu, counter.cov, matched, counter.rf)
            simulated_annealing(probe, b, seed=cfg.seed + 100 + k)
            if probe.best_selection:
                candidates.append(sorted(probe.best_selection))

    elif cfg.tier2 == "none":
        rng = np.random.default_rng(cfg.seed + 11)
        candidates = [sorted(rng.choice(n, size=b, replace=False).tolist())]

    else:
        raise ValueError(f"unknown tier2 mode: {cfg.tier2!r}")

    return candidates, diag


def run_pipeline(mu: np.ndarray, cov: np.ndarray, cfg: PipelineConfig,
                 rf: float = 0.0) -> PipelineResult:
    started = time.perf_counter()
    n = mu.size
    counter = EvalCounter(mu, cov, cfg.eval_budget, rf)

    candidates, diag = _tier2_candidates(counter, cfg, n)

    if not diag["simulable"]:
        return PipelineResult(selection=[], sharpe=float("nan"), n_evals=0,
                              seconds=time.perf_counter() - started,
                              seed_source="not-simulable", simulable=False)

    seed_source = cfg.tier2
    try:
        if candidates:
            seed = max(candidates, key=counter)
        else:
            rng = np.random.default_rng(cfg.seed + 23)
            seed = sorted(rng.choice(n, size=cfg.budget, replace=False).tolist())
            seed_source = "fallback-random"
            counter(seed)
        final = local_refine(counter, seed, n) if cfg.tier3 else sorted(seed)
    except BudgetExhausted:
        final = sorted(counter.best_selection)

    if cfg.tier3:
        final = sorted(counter.best_selection) if counter.best_selection else final
        value = counter.best_value
    else:
        value = sharpe(mu, cov, final, rf)

    mean = lambda xs: float(np.mean(xs)) if xs else float("nan")
    return PipelineResult(
        selection=list(final), sharpe=float(value), n_evals=counter.n_evals,
        seconds=time.perf_counter() - started, seed_source=seed_source,
        qaoa_seconds=diag["qaoa_seconds"], qaoa_surrogate_evals=diag["surrogate_evals"],
        feasible_fraction=mean(diag["feasible"]), approx_ratio=mean(diag["approx"]),
        p_optimal=mean(diag["p_opt"]), trajectory=list(counter.trajectory))


# --- single-instance demonstration --------------------------------------------
_ground_sharpe = _exact["sharpe"]
_demo_result = run_pipeline(_mu, _cov, PipelineConfig(seed=1))
print("QuOpt-Agents on one bull-regime instance")
print(f"  selection        {_demo_result.selection}")
print(f"  Sharpe           {_demo_result.sharpe:.6f}")
print(f"  exact optimum    {_ground_sharpe:.6f}")
print(f"  optimality gap   {100 * (_ground_sharpe - _demo_result.sharpe) / abs(_ground_sharpe):.4f} %")
print(f"  seed source      {_demo_result.seed_source}")
print(f"  f-evaluations    {_demo_result.n_evals:,}")
print(f"  QAOA wall-clock  {_demo_result.qaoa_seconds:.2f} s over "
      f"{len(PipelineConfig().tier1_spectrum)} surrogates")
print(f"  feasible shots   {100 * _demo_result.feasible_fraction:.1f} %")
print(f"  approx ratio     {_demo_result.approx_ratio:.4f}")


## 10. Experiments

All results land in one long-format table, `RESULTS`, with one row per
(experiment, instance, method). Every manuscript table is then a view over that table, so a
number cannot appear in two tables with two different values — which is how the inconsistencies
flagged by Editor 2 arose in the first place.

Each row carries both `sharpe_is` (estimation window, the quantity being optimised) and
`sharpe_oos` (evaluation window, never seen by the optimiser). Reporting only the former, as
the submission did, measures fitting rather than performance. Reporting both is also
scientifically interesting: at this problem size the out-of-sample ordering is dominated by
estimation error, not by which optimiser was used, and that is worth saying in the paper.

`gap_pct` is the optimality gap against exhaustive enumeration, in percent of the optimal
Sharpe. `is_optimal` flags rows that attained the exact optimum.


In [ ]:
PROPOSED = "QuOpt-Agents"
ENUMERATION = "Exhaustive Enumeration"

RECORDS: List[Dict[str, object]] = []
TRAJECTORIES: Dict[Tuple[str, str], np.ndarray] = {}


def record(experiment: str, instance: str, method: str, selection: Sequence[int],
           mu_is: np.ndarray, cov_is: np.ndarray,
           mu_oos: Optional[np.ndarray], cov_oos: Optional[np.ndarray],
           n_evals: int, seconds: float, n: int, b: int,
           optimum: Optional[float] = None, regime: str = "",
           **extra) -> None:
    ret_is, vol_is = portfolio_moments(mu_is, cov_is, selection)
    s_is = sharpe(mu_is, cov_is, selection) if selection else float("nan")
    if mu_oos is not None and selection:
        ret_oos, vol_oos = portfolio_moments(mu_oos, cov_oos, selection)
        s_oos = sharpe(mu_oos, cov_oos, selection)
    else:
        ret_oos = vol_oos = s_oos = float("nan")
    gap = float("nan")
    is_opt = ""
    if optimum is not None and selection and np.isfinite(s_is):
        gap = 100.0 * (optimum - s_is) / abs(optimum)
        is_opt = bool(abs(optimum - s_is) < 1e-9)
    row = {"experiment": experiment, "instance": instance, "regime": regime,
           "method": method, "n": n, "B": b,
           "selection": ",".join(str(i) for i in selection),
           "sharpe_is": s_is, "sharpe_oos": s_oos,
           "ret_is": ret_is, "vol_is": vol_is, "ret_oos": ret_oos, "vol_oos": vol_oos,
           "gap_pct": gap, "is_optimal": is_opt,
           "n_evals": n_evals, "seconds": seconds}
    row.update(extra)
    RECORDS.append(row)


def evaluate_instance(experiment: str, instance: str, mu_is: np.ndarray,
                      cov_is: np.ndarray, mu_oos: Optional[np.ndarray],
                      cov_oos: Optional[np.ndarray], seed: int,
                      regime: str = "", keep_trajectories: bool = False,
                      cfg: Config = CFG) -> None:
    """Run enumeration, the proposed pipeline, and every baseline on one instance."""
    n, b = mu_is.size, cfg.budget

    exact = enumerate_optimum(mu_is, cov_is, b)
    optimum = exact["sharpe"] if exact else None
    if exact:
        record(experiment, instance, ENUMERATION, exact["selection"], mu_is, cov_is,
               mu_oos, cov_oos, exact["n_evals"], exact["seconds"], n, b,
               optimum, regime)

    result = run_pipeline(mu_is, cov_is,
                          PipelineConfig(budget=b, eval_budget=cfg.eval_budget,
                                         qaoa=cfg.qaoa, seed=seed))
    record(experiment, instance, PROPOSED, result.selection, mu_is, cov_is,
           mu_oos, cov_oos, result.n_evals, result.seconds, n, b, optimum, regime,
           qaoa_seconds=result.qaoa_seconds,
           feasible_fraction=result.feasible_fraction,
           approx_ratio=result.approx_ratio, p_optimal=result.p_optimal,
           seed_source=result.seed_source)
    if keep_trajectories and result.trajectory:
        counter_like = EvalCounter(mu_is, cov_is, cfg.eval_budget)
        counter_like.trajectory = result.trajectory
        TRAJECTORIES[(PROPOSED, instance)] = counter_like.curve(cfg.eval_budget)

    for name, fn in BASELINES.items():
        counter = EvalCounter(mu_is, cov_is, cfg.eval_budget)
        t0 = time.perf_counter()
        selection = fn(counter, b, seed=seed)
        elapsed = time.perf_counter() - t0
        record(experiment, instance, name, selection, mu_is, cov_is, mu_oos, cov_oos,
               counter.n_evals, elapsed, n, b, optimum, regime)
        if keep_trajectories:
            TRAJECTORIES[(name, instance)] = counter.curve(cfg.eval_budget)


# --- synthetic multi-regime study ---------------------------------------------
def run_synthetic(cfg: Config = CFG) -> None:
    schedule = regime_schedule(cfg)
    print(f"[synthetic] {len(schedule)} scenarios, N={cfg.n_assets}, B={cfg.budget}")
    started = time.perf_counter()
    for k, (regime, seed) in enumerate(schedule, start=1):
        panel = scenario_panel(regime, seed, cfg)
        evaluate_instance("synthetic", f"S{k:03d}", panel["mu_is"], panel["cov_is"],
                          panel["mu_oos"], panel["cov_oos"], seed=seed, regime=regime,
                          keep_trajectories=(k == 1), cfg=cfg)
        if k % 5 == 0 or k == len(schedule):
            print(f"  {k}/{len(schedule)} scenarios "
                  f"({time.perf_counter() - started:.0f} s elapsed)")
    print(f"[synthetic] done in {time.perf_counter() - started:.0f} s")


run_synthetic()
print(f"\nrows so far: {len(RECORDS)}")


In [ ]:
def run_real(cfg: Config = CFG) -> None:
    print(f"[real] {len(REAL_WINDOWS)} walk-forward windows, "
          f"{REAL_RETURNS.shape[1]} assets, {len(REAL_RETURNS)} observations")
    if REAL_DATA_IS_SYNTHETIC:
        print("       WARNING: running on the synthetic surrogate, not Yahoo Finance data")
    started = time.perf_counter()
    for window in REAL_WINDOWS:
        train = REAL_RETURNS.iloc[window.train_start:window.train_stop]
        test = REAL_RETURNS.iloc[window.test_start:window.test_stop]
        mu_is, cov_is = estimate_moments(train)
        mu_oos, cov_oos = estimate_moments(test)
        label = f"W{window.index + 1:02d}"
        evaluate_instance("real", label, mu_is, cov_is, mu_oos, cov_oos,
                          seed=500 + window.index, regime="real",
                          keep_trajectories=(window.index == 0), cfg=cfg)
        if (window.index + 1) % 5 == 0 or window.index == len(REAL_WINDOWS) - 1:
            print(f"  {window.index + 1}/{len(REAL_WINDOWS)} windows "
                  f"({time.perf_counter() - started:.0f} s elapsed)")
    print(f"[real] done in {time.perf_counter() - started:.0f} s")


run_real()
print(f"\nrows so far: {len(RECORDS)}")


## 11. Ablations

### 11.1 Tier-2 substitution (Editor 1, Comment 2)

This is the experiment that answers "what does the quantum component actually contribute?".
The architecture is held fixed and only the Tier-2 seed generator is swapped:

| Arm | Tier-2 mechanism |
|---|---|
| `qaoa` | QAOA sampling, as specified in Section 5 |
| `sa` | simulated annealing given the *same* number of surrogate evaluations QAOA's outer loop consumes |
| `greedy` | deterministic top-`B` on a variance-tilted return vector |
| `random` | uniformly random feasible portfolio |
| `none` | no Tier 2; Tier 3 starts from a random portfolio |

If `qaoa` is statistically indistinguishable from `sa` and `greedy` after Tier 3, then the
quantum layer is not carrying the result at this problem size, and the manuscript must say so.
That is the expected outcome and it is a legitimate, publishable finding — provided it is
reported rather than concealed by an unfair baseline.

### 11.2 Tier removal (manuscript Table 9)

Six configurations isolating each tier's marginal contribution: the full stack, minus Tier 1
(single risk level instead of the spectrum), minus Tier 3 (raw QAOA seed), Tier 2 alone,
Tier 3 alone from a random start, and Tiers 1+3 with no quantum layer at all.

Because QAOA dominates the runtime, the tier ablation is run on the first `N_ABLATION`
scenarios rather than all of them. The count is reported in the table note so the sample
size behind every ablation number is explicit.


In [ ]:
N_ABLATION = min(20, CFG.n_scenarios)

TIER2_ARMS = {
    "Tier-2 = QAOA": dict(tier2="qaoa", tier3=True),
    "Tier-2 = Simulated Annealing": dict(tier2="sa", tier3=True),
    "Tier-2 = Greedy tilt": dict(tier2="greedy", tier3=True),
    "Tier-2 = Random": dict(tier2="random", tier3=True),
    "Tier-2 removed": dict(tier2="none", tier3=True),
}

TIER_CONFIGS = {
    "Full (T1+T2+T3)": dict(tier1_spectrum=(0.25, 0.5, 1.0), tier2="qaoa", tier3=True),
    "No Tier 1 (single q)": dict(tier1_spectrum=(0.5,), tier2="qaoa", tier3=True),
    "No Tier 3 (raw QAOA)": dict(tier1_spectrum=(0.25, 0.5, 1.0), tier2="qaoa", tier3=False),
    "Tier 2 only": dict(tier1_spectrum=(0.5,), tier2="qaoa", tier3=False),
    "Tier 3 only": dict(tier1_spectrum=(0.5,), tier2="none", tier3=True),
    "No quantum (T1+T3)": dict(tier1_spectrum=(0.25, 0.5, 1.0), tier2="greedy", tier3=True),
}


def _ablation_panels(cfg: Config = CFG) -> List[Tuple[str, str, Dict]]:
    out = []
    for k, (regime, seed) in enumerate(regime_schedule(cfg)[:N_ABLATION], start=1):
        out.append((f"S{k:03d}", regime, scenario_panel(regime, seed, cfg)))
    return out


def run_ablations(cfg: Config = CFG) -> None:
    panels = _ablation_panels(cfg)
    for experiment, arms in (("tier2_ablation", TIER2_ARMS),
                             ("tier_ablation", TIER_CONFIGS)):
        print(f"[{experiment}] {len(arms)} arms x {len(panels)} scenarios")
        started = time.perf_counter()
        for label, kwargs in arms.items():
            for instance, regime, panel in panels:
                exact = enumerate_optimum(panel["mu_is"], panel["cov_is"], cfg.budget)
                result = run_pipeline(
                    panel["mu_is"], panel["cov_is"],
                    PipelineConfig(budget=cfg.budget, eval_budget=cfg.eval_budget,
                                   qaoa=cfg.qaoa, seed=panel["seed"], **kwargs))
                record(experiment, instance, label, result.selection,
                       panel["mu_is"], panel["cov_is"], panel["mu_oos"],
                       panel["cov_oos"], result.n_evals, result.seconds,
                       cfg.n_assets, cfg.budget,
                       exact["sharpe"] if exact else None, regime,
                       qaoa_seconds=result.qaoa_seconds,
                       feasible_fraction=result.feasible_fraction,
                       approx_ratio=result.approx_ratio)
            print(f"  {label}: done ({time.perf_counter() - started:.0f} s cumulative)")
        print(f"[{experiment}] done in {time.perf_counter() - started:.0f} s")


run_ablations()
print(f"\nrows so far: {len(RECORDS)}")


## 12. Scale-up: where each method stops working

This section supplies the "when would the quantum layer matter?" half of the answer to
Editor 1's Comment 2, and it does so by reporting failure honestly rather than extrapolating.

Three walls are hit at different sizes:

- **Exhaustive enumeration** becomes intractable once `C(N,B)` exceeds a few million.
- **Exact state-vector QAOA simulation** becomes intractable at roughly 22 qubits, because
  the state vector holds `2^N` complex amplitudes and the per-iteration cost is `O(N * 2^N)`.
  Beyond the ceiling the row is marked `not simulable` — the submitted code would have
  silently fallen back to a classical solution and reported it as a quantum result.
- **Classical metaheuristics** keep running at every size tested, which is precisely the
  point: nothing in this study establishes a regime where the quantum layer is necessary.
  The manuscript should state that a hardware experiment, not a larger simulation, is what
  would settle the question.


In [ ]:
SCALEUP_METHODS = {
    "Strong Greedy": strong_greedy,
    "Genetic Algorithm": genetic_algorithm,
    "Simulated Annealing": simulated_annealing,
}


def run_scaleup(cfg: Config = CFG) -> None:
    print(f"[scaleup] {len(cfg.scaleup)} instances "
          f"(QAOA ceiling {cfg.qaoa.max_qubits} qubits, "
          f"enumeration ceiling {ENUM_LIMIT:,} portfolios)")
    for n_assets, b in cfg.scaleup:
        label = f"N={n_assets},B={b}"
        total = math.comb(n_assets, b)
        sim = simulate_regime("bull", n_assets, cfg.estim_days + 1, seed=9000 + n_assets)
        mu_is, cov_is = estimate_moments(log_returns(sim["prices"]))

        exact = enumerate_optimum(mu_is, cov_is, b)
        optimum = exact["sharpe"] if exact else None
        if exact:
            record("scaleup", label, ENUMERATION, exact["selection"], mu_is, cov_is,
                   None, None, exact["n_evals"], exact["seconds"], n_assets, b, optimum)
        else:
            record("scaleup", label, ENUMERATION, [], mu_is, cov_is, None, None,
                   0, float("nan"), n_assets, b, None, note="intractable")

        result = run_pipeline(mu_is, cov_is,
                              PipelineConfig(budget=b, eval_budget=cfg.eval_budget,
                                             qaoa=cfg.qaoa, seed=9000 + n_assets))
        record("scaleup", label, PROPOSED, result.selection, mu_is, cov_is, None, None,
               result.n_evals, result.seconds, n_assets, b, optimum,
               qaoa_seconds=result.qaoa_seconds,
               feasible_fraction=result.feasible_fraction,
               approx_ratio=result.approx_ratio,
               note="" if result.simulable else "not simulable",
               simulable=result.simulable)

        for name, fn in SCALEUP_METHODS.items():
            counter = EvalCounter(mu_is, cov_is, cfg.eval_budget)
            t0 = time.perf_counter()
            selection = fn(counter, b, seed=9000 + n_assets)
            record("scaleup", label, name, selection, mu_is, cov_is, None, None,
                   counter.n_evals, time.perf_counter() - t0, n_assets, b, optimum)

        status = ("simulated" if result.simulable
                  else f"NOT simulable (>{cfg.qaoa.max_qubits} qubits)")
        enum_status = f"{total:,} portfolios" if exact else f"{total:,} intractable"
        print(f"  {label}: {enum_status}; QAOA {status}")
    print("[scaleup] done")


run_scaleup()

RESULTS = pd.DataFrame(RECORDS)
RESULTS.to_csv(OUTDIR / "results" / "results.csv", index=False)
print(f"\n{len(RESULTS)} rows written to {OUTDIR / 'results' / 'results.csv'}")
print(RESULTS.groupby("experiment").size().to_string())


## 13. Statistical analysis (Editor 2, Comments 2 and 3)

Four specific errors in the submitted analysis are corrected here.

**Ties were counted as wins.** `(a > b).mean()` reports a win whenever the two Sharpe ratios
differ in the last floating-point bit. When both methods find the same optimum — the normal
case at this problem size — that comparison is reporting numerical noise. Wins, losses and
ties are now counted separately with an explicit tolerance of `1e-9`, and the tie count is
reported. This alone is likely to move the headline win rate substantially, and the honest
number is the one to publish.

**The Wilcoxon statistic exceeded its own upper bound.** For `n` non-zero paired differences
the signed-rank statistic satisfies `0 <= W <= n(n+1)/2`. The submitted value was outside
that range, which means it was not the statistic it was labelled as. `wilcoxon_report` below
computes `W` with SciPy, recomputes the bound, and **raises** if the bound is violated, so
this class of error cannot survive a rerun. The zero-difference (tied) pairs are dropped
before ranking, per Wilcoxon's original procedure, and the effective `n` after dropping is
reported alongside `W` — omitting it was what made the original value unverifiable.

**Confidence intervals were mismatched to the test.** The submission reported separate
marginal intervals for each method's mean Sharpe and then remarked on their overlap as if it
contradicted the significant paired test. It does not: a paired test concerns the
*distribution of within-scenario differences*, whose variance is
`var(A) + var(B) - 2cov(A,B)`. With strongly positively correlated methods — and these are
near-perfectly correlated, since they solve the same instances — that variance is far smaller
than either marginal variance. The interval that belongs with a paired test is the interval on
the mean difference, and that is what is reported. Marginal intervals are shown too, but
labelled as descriptive.

**"Improvement %" was ambiguous.** The ratio of mean Sharpes and the mean of per-scenario
ratios are different statistics and the submission mixed them between the abstract and the
tables. Both are computed here, side by side, with the difference between them visible.

Effect size is Cohen's `d` for paired samples, `mean(diff) / sd(diff)`, which is the
convention that matches a paired `t`-test; using the pooled cross-sectional standard
deviation instead would inflate `d` here by roughly the correlation factor.


In [ ]:
TIE_TOL = 1e-9


class InconsistentStatistic(AssertionError):
    """Raised when a reported statistic violates its own definitional bound."""


def wilcoxon_report(diff: np.ndarray) -> Dict[str, object]:
    """Wilcoxon signed-rank test with an enforced bound check on W."""
    diff = np.asarray(diff, dtype=float)
    diff = diff[np.isfinite(diff)]
    nonzero = diff[np.abs(diff) > TIE_TOL]
    n_eff = nonzero.size
    bound = n_eff * (n_eff + 1) / 2

    if n_eff == 0:
        return {"W": float("nan"), "W_max": 0.0, "n_eff": 0,
                "p": float("nan"), "method": "undefined (all pairs tied)"}

    method = "exact" if n_eff <= 25 else "approx"
    res = sps.wilcoxon(nonzero, zero_method="wilcox", alternative="two-sided",
                       method=method)
    w_stat = float(res.statistic)
    if not (0.0 - 1e-9 <= w_stat <= bound + 1e-9):
        raise InconsistentStatistic(
            f"W={w_stat} lies outside [0, n(n+1)/2] = [0, {bound}] for n_eff={n_eff}")
    return {"W": w_stat, "W_max": bound, "n_eff": n_eff,
            "p": float(res.pvalue), "method": method}


def paired_report(a: np.ndarray, b: np.ndarray, label_a: str, label_b: str,
                  alpha: float = 0.05) -> Dict[str, object]:
    """Complete paired comparison of two methods over the same instances."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    keep = np.isfinite(a) & np.isfinite(b)
    a, b = a[keep], b[keep]
    n = a.size
    diff = a - b

    wins = int(np.sum(diff > TIE_TOL))
    losses = int(np.sum(diff < -TIE_TOL))
    ties = int(n - wins - losses)

    mean_diff = float(diff.mean())
    sd_diff = float(diff.std(ddof=1)) if n > 1 else 0.0
    se_diff = sd_diff / math.sqrt(n) if n > 1 else 0.0
    t_crit = float(sps.t.ppf(1 - alpha / 2, n - 1)) if n > 1 else float("nan")

    if sd_diff > TIE_TOL:
        t_stat, p_t = sps.ttest_rel(a, b)
        t_stat, p_t = float(t_stat), float(p_t)
        cohens_d = mean_diff / sd_diff
    else:
        t_stat = p_t = float("nan")
        cohens_d = 0.0

    try:
        wil = wilcoxon_report(diff)
    except InconsistentStatistic as exc:
        raise
    corr = float(np.corrcoef(a, b)[0, 1]) if n > 1 and a.std() > 0 and b.std() > 0 \
        else float("nan")

    return {
        "comparison": f"{label_a} vs {label_b}", "n": n,
        "mean_a": float(a.mean()), "mean_b": float(b.mean()),
        "sd_a": float(a.std(ddof=1)) if n > 1 else 0.0,
        "sd_b": float(b.std(ddof=1)) if n > 1 else 0.0,
        "corr_ab": corr,
        "wins": wins, "losses": losses, "ties": ties,
        "win_rate_excl_ties": (wins / (wins + losses)) if (wins + losses) else float("nan"),
        "mean_diff": mean_diff, "sd_diff": sd_diff,
        "ci_diff_lo": mean_diff - t_crit * se_diff if n > 1 else float("nan"),
        "ci_diff_hi": mean_diff + t_crit * se_diff if n > 1 else float("nan"),
        "t_stat": t_stat, "p_ttest": p_t, "cohens_d": cohens_d,
        "W": wil["W"], "W_max": wil["W_max"], "n_eff": wil["n_eff"],
        "p_wilcoxon": wil["p"], "wilcoxon_method": wil["method"],
        # Two distinct definitions of "improvement", both reported.
        "improve_ratio_of_means": 100 * (a.mean() - b.mean()) / abs(b.mean())
        if abs(b.mean()) > 0 else float("nan"),
        "improve_mean_of_ratios": 100 * float(np.mean(
            np.where(np.abs(b) > 1e-12, (a - b) / np.abs(b), np.nan)))
        if n else float("nan"),
    }


def marginal_ci(x: np.ndarray, alpha: float = 0.05) -> Tuple[float, float]:
    """Descriptive interval on a single method's mean. NOT the paired-test interval."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size < 2:
        return float("nan"), float("nan")
    se = x.std(ddof=1) / math.sqrt(x.size)
    half = float(sps.t.ppf(1 - alpha / 2, x.size - 1)) * se
    return float(x.mean() - half), float(x.mean() + half)


def pivot_metric(frame: pd.DataFrame, experiment: str,
                 metric: str = "sharpe_is") -> pd.DataFrame:
    sub = frame[frame["experiment"] == experiment]
    return sub.pivot_table(index="instance", columns="method", values=metric)


def comparison_table(frame: pd.DataFrame, experiment: str, reference: str = PROPOSED,
                     metric: str = "sharpe_is") -> pd.DataFrame:
    wide = pivot_metric(frame, experiment, metric)
    if reference not in wide.columns:
        return pd.DataFrame()
    rows = [paired_report(wide[reference].to_numpy(), wide[other].to_numpy(),
                          reference, other)
            for other in wide.columns if other != reference]
    return pd.DataFrame(rows)


STATS_IS = comparison_table(RESULTS, "synthetic", metric="sharpe_is")
STATS_OOS = comparison_table(RESULTS, "synthetic", metric="sharpe_oos")
STATS_REAL = comparison_table(RESULTS, "real", metric="sharpe_oos")

pd.set_option("display.width", 200, "display.max_columns", 40)
_cols = ["comparison", "n", "wins", "losses", "ties", "mean_diff",
         "ci_diff_lo", "ci_diff_hi", "t_stat", "p_ttest", "cohens_d",
         "W", "W_max", "n_eff", "p_wilcoxon"]
print("Synthetic study, IN-SAMPLE Sharpe (the quantity being optimised)")
print(STATS_IS[_cols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nSynthetic study, OUT-OF-SAMPLE Sharpe (never seen by the optimiser)")
print(STATS_OOS[_cols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nPASS  every reported W satisfies 0 <= W <= n(n+1)/2")


### 13.1 Why overlapping marginal intervals do not contradict a significant paired test

The cell below demonstrates the point on the actual data rather than asserting it. It prints,
for each baseline, the marginal 95% interval for both methods, whether those intervals
overlap, the correlation between the two methods across scenarios, the paired-difference
interval, and the paired `p`-value.

The pattern to expect — and the pattern the manuscript should describe — is heavy overlap of
the marginal intervals combined with a paired-difference interval that may exclude zero,
driven by a between-method correlation close to 1. The submitted text treated the overlap as
a caveat undermining the significance claim; the correct reading is that the two intervals
answer different questions.

## 14. Manuscript tables

Every table is a view over `RESULTS` and is written to `tables/` as both CSV and Markdown,
each carrying a note that states its sample size and any caveat. The note travels with the
table so that a number cannot be quoted in the paper without its qualification.


In [ ]:
_wide_is = pivot_metric(RESULTS, "synthetic", "sharpe_is")
_rows = []
for _other in _wide_is.columns:
    if _other == PROPOSED:
        continue
    _a, _b = _wide_is[PROPOSED].to_numpy(), _wide_is[_other].to_numpy()
    _rep = paired_report(_a, _b, PROPOSED, _other)
    _lo_a, _hi_a = marginal_ci(_a)
    _lo_b, _hi_b = marginal_ci(_b)
    _rows.append({
        "baseline": _other,
        "marginal CI (proposed)": f"[{_lo_a:.3f}, {_hi_a:.3f}]",
        "marginal CI (baseline)": f"[{_lo_b:.3f}, {_hi_b:.3f}]",
        "overlap": "yes" if (_lo_a <= _hi_b and _lo_b <= _hi_a) else "no",
        "corr(A,B)": round(_rep["corr_ab"], 4),
        "paired diff CI": f"[{_rep['ci_diff_lo']:+.5f}, {_rep['ci_diff_hi']:+.5f}]",
        "excludes 0": "yes" if (_rep["ci_diff_lo"] > 0 or _rep["ci_diff_hi"] < 0) else "no",
        "p (paired t)": f"{_rep['p_ttest']:.3g}",
    })
CI_DEMO = pd.DataFrame(_rows)
print("Marginal intervals vs the paired-difference interval (in-sample Sharpe)")
print(CI_DEMO.to_string(index=False))


In [ ]:
METHOD_ORDER = [ENUMERATION, PROPOSED] + list(BASELINES)


def _order(frame: pd.DataFrame, column: str = "method") -> pd.DataFrame:
    known = [m for m in METHOD_ORDER if m in set(frame[column])]
    rest = sorted(set(frame[column]) - set(known))
    frame = frame.set_index(column).reindex(known + rest).reset_index()
    return frame


def aggregate_by_method(frame: pd.DataFrame, experiment: str) -> pd.DataFrame:
    sub = frame[frame["experiment"] == experiment]
    rows = []
    for method, block in sub.groupby("method"):
        lo, hi = marginal_ci(block["sharpe_is"].to_numpy())
        gaps = block["gap_pct"].dropna().to_numpy()
        rows.append({
            "method": method,
            "n_instances": len(block),
            "sharpe_is_mean": block["sharpe_is"].mean(),
            "sharpe_is_sd": block["sharpe_is"].std(ddof=1),
            "sharpe_is_ci": f"[{lo:.4f}, {hi:.4f}]",
            "sharpe_oos_mean": block["sharpe_oos"].mean(),
            "sharpe_oos_sd": block["sharpe_oos"].std(ddof=1),
            "ret_is_mean": block["ret_is"].mean(),
            "vol_is_mean": block["vol_is"].mean(),
            "gap_mean_pct": gaps.mean() if gaps.size else float("nan"),
            "gap_max_pct": gaps.max() if gaps.size else float("nan"),
            "pct_optimal": 100.0 * np.mean(gaps < 1e-9) if gaps.size else float("nan"),
            "f_evals_mean": block["n_evals"].mean(),
            "seconds_mean": block["seconds"].mean(),
        })
    return _order(pd.DataFrame(rows))


def ablation_summary(frame: pd.DataFrame, experiment: str) -> pd.DataFrame:
    sub = frame[frame["experiment"] == experiment]
    if sub.empty:
        return pd.DataFrame()
    base = sub[sub["method"].isin(["Tier-2 = QAOA", "Full (T1+T2+T3)"])]
    reference = base["sharpe_is"].mean() if not base.empty else float("nan")
    rows = []
    for arm, block in sub.groupby("method"):
        gaps = block["gap_pct"].dropna().to_numpy()
        rows.append({
            "configuration": arm,
            "n": len(block),
            "sharpe_is_mean": block["sharpe_is"].mean(),
            "sharpe_is_sd": block["sharpe_is"].std(ddof=1),
            "sharpe_oos_mean": block["sharpe_oos"].mean(),
            "gap_mean_pct": gaps.mean() if gaps.size else float("nan"),
            "pct_optimal": 100.0 * np.mean(gaps < 1e-9) if gaps.size else float("nan"),
            "delta_vs_full_pct": 100.0 * (block["sharpe_is"].mean() - reference)
            / abs(reference) if np.isfinite(reference) and reference else float("nan"),
            "f_evals_mean": block["n_evals"].mean(),
            "seconds_mean": block["seconds"].mean(),
        })
    return pd.DataFrame(rows).sort_values("sharpe_is_mean", ascending=False)


def build_tables() -> Dict[str, Tuple[pd.DataFrame, str]]:
    tables: Dict[str, Tuple[pd.DataFrame, str]] = {}
    n_syn = RESULTS[RESULTS.experiment == "synthetic"]["instance"].nunique()
    n_real = RESULTS[RESULTS.experiment == "real"]["instance"].nunique()

    tables["table03_aggregate_synthetic"] = (
        aggregate_by_method(RESULTS, "synthetic"),
        f"Synthetic multi-regime study. N={CFG.n_assets}, B={CFG.budget}, "
        f"{n_syn} scenarios across {len(REGIMES)} regimes. Every stochastic searcher "
        f"received {CFG.eval_budget:,} objective evaluations. 'gap' is the shortfall "
        f"against exhaustive enumeration over all "
        f"{math.comb(CFG.n_assets, CFG.budget):,} feasible portfolios. Confidence "
        f"intervals are descriptive marginal intervals; see the paired table for "
        f"inference.")

    tables["table04_real_walkforward"] = (
        aggregate_by_method(RESULTS, "real"),
        f"Real-data walk-forward study. {n_real} windows, {CFG.lookback}-day "
        f"estimation and {CFG.horizon}-day evaluation, non-overlapping evaluation "
        f"windows. "
        + ("SYNTHETIC SURROGATE - Yahoo Finance was unreachable at run time; these "
           "numbers must be regenerated with network access before submission."
           if REAL_DATA_IS_SYNTHETIC else
           f"Assets: {', '.join(CFG.tickers)}. Period {CFG.start} to {CFG.end}, "
           f"daily adjusted closes from Yahoo Finance."))

    regime_block = RESULTS[(RESULTS.experiment == "synthetic")
                           & (RESULTS.method.isin([PROPOSED, ENUMERATION,
                                                   "Strong Greedy", "Classical MVO"]))]
    tables["table05_by_regime"] = (
        regime_block.pivot_table(index="regime", columns="method",
                                 values=["sharpe_is", "sharpe_oos"],
                                 aggfunc="mean").round(4).reset_index(),
        "Mean Sharpe ratio by market regime, in-sample and out-of-sample. Scenario "
        "counts per regime are fixed by design at "
        + ", ".join(f"{k} {v.count}" for k, v in REGIMES.items())
        + f" (scaled to {n_syn} scenarios), not drawn at random.")

    for key, block, metric in (("table06_paired_synthetic_is", STATS_IS, "in-sample"),
                               ("table07_paired_synthetic_oos", STATS_OOS,
                                "out-of-sample"),
                               ("table08_paired_real_oos", STATS_REAL,
                                "real-data out-of-sample")):
        if not block.empty:
            tables[key] = (block.round(6), (
                f"Paired comparison of {PROPOSED} against each baseline on "
                f"{metric} Sharpe. 'ties' counts pairs differing by less than "
                f"{TIE_TOL:g}; the submitted analysis counted these as wins. "
                f"'W_max' = n_eff(n_eff+1)/2 is the definitional upper bound on the "
                f"Wilcoxon statistic and n_eff excludes tied pairs. The interval is on "
                f"the mean paired difference, which is the interval that belongs with a "
                f"paired test."))

    tables["table09_ci_interpretation"] = (
        CI_DEMO,
        "Marginal intervals versus the paired-difference interval. Overlap of the "
        "marginal intervals is expected and does not bear on the paired test, because "
        "the methods are almost perfectly correlated across scenarios: the variance of "
        "the difference is var(A)+var(B)-2cov(A,B), far below either marginal variance.")

    tables["table10_tier2_ablation"] = (
        ablation_summary(RESULTS, "tier2_ablation").round(6),
        f"Tier-2 substitution over the first {N_ABLATION} synthetic scenarios. Only the "
        f"Tier-2 seed generator changes; Tiers 1 and 3 are identical across arms. The "
        f"'sa' arm receives the same number of surrogate evaluations as QAOA's outer "
        f"loop. This is the experiment that isolates the quantum layer's contribution.")

    tables["table11_tier_ablation"] = (
        ablation_summary(RESULTS, "tier_ablation").round(6),
        f"Tier-removal ablation over the first {N_ABLATION} synthetic scenarios. "
        f"'delta_vs_full' is relative to the full three-tier stack.")

    scale = RESULTS[RESULTS.experiment == "scaleup"].copy()
    tables["table12_scaleup"] = (
        scale.pivot_table(index="instance", columns="method",
                          values=["sharpe_is", "seconds"], aggfunc="first").reset_index(),
        f"Scale-up sweep. Exhaustive enumeration is attempted up to {ENUM_LIMIT:,} "
        f"feasible portfolios; exact state-vector QAOA up to "
        f"{CFG.qaoa.max_qubits} qubits. Blank QAOA entries are instances that cannot be "
        f"simulated at all, reported as such rather than silently replaced by a "
        f"classical result.")

    qaoa_rows = RESULTS[(RESULTS.method == PROPOSED)
                        & RESULTS.get("feasible_fraction",
                                      pd.Series(dtype=float)).notna()]
    if not qaoa_rows.empty:
        tables["table13_qaoa_diagnostics"] = (
            qaoa_rows.groupby("experiment").agg(
                instances=("instance", "nunique"),
                feasible_shots_pct=("feasible_fraction", lambda s: 100 * s.mean()),
                approx_ratio=("approx_ratio", "mean"),
                p_optimal=("p_optimal", "mean"),
                qaoa_seconds=("qaoa_seconds", "mean")).round(4).reset_index(),
            f"QAOA diagnostics, averaged over instances and over the "
            f"{len(PipelineConfig().tier1_spectrum)} surrogates in the Tier-1 spectrum. "
            f"'feasible_shots_pct' is the share of the {CFG.qaoa.shots} measurements "
            f"holding exactly B assets. 'approx_ratio' and 'p_optimal' are computed "
            f"against the ground state of the feasible subspace.")

    return tables


def _to_markdown(frame: pd.DataFrame) -> str:
    try:
        return frame.to_markdown(index=False)
    except ImportError:            # tabulate is optional
        return "```\n" + frame.to_string(index=False) + "\n```"


TABLES = build_tables()
for _name, (_df, _note) in TABLES.items():
    _df.to_csv(OUTDIR / "tables" / f"{_name}.csv", index=False)
    (OUTDIR / "tables" / f"{_name}.md").write_text(
        f"# {_name}\n\n{_to_markdown(_df)}\n\n_Note._ {_note}\n", encoding="utf-8")

print(f"{len(TABLES)} tables written to {OUTDIR / 'tables'}\n")
for _name in ("table03_aggregate_synthetic", "table10_tier2_ablation",
              "table11_tier_ablation", "table12_scaleup"):
    _df, _note = TABLES[_name]
    print("=" * 100)
    print(_name)
    print("=" * 100)
    print(_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print(f"\nNote. {_note}\n")


## 15. Figures

Six figures, each exported at 600 dpi PNG and as vector PDF. Presentation fixes requested by
the reviewers:

**Figure 1 (architecture).** Redrawn from a single specification so that box sizes, fonts and
arrow styles are uniform, and so that the tier labels in the figure match the tier names used
in the text. Editor 1's Comment 1 objected to inconsistent elements; the cause was that the
original was assembled by hand rather than generated.

**Figure 3 (convergence) — Editor 2, Comment 4.** The submitted figure plotted only the points
at which a method improved, so a method that converged in a few dozen evaluations rendered as a
curve that appeared to vanish. Three changes: the x-axis is now cumulative objective
evaluations (a common unit across methods, unlike "iterations", which means different things
to a GA and to simulated annealing); curves are drawn as step functions carried forward to the
budget; and the x-axis is logarithmic so early convergence is visible instead of being
compressed against the origin.

**Figure 4 (risk-return).** Legend moved outside the axes, markers differentiated by shape as
well as colour so the figure survives greyscale printing, and the exact optimum marked
explicitly so the reader can see how close every method is.

**Figure 6 (optimality gap)** is new and is the figure the paper's central claim now rests on.


In [ ]:
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 600, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":",
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.constrained_layout.use": True,
})

PALETTE = {
    ENUMERATION: "#111111", PROPOSED: "#1f6fb2",
    "Classical MVO": "#b03060", "Strong Greedy": "#e08a1e",
    "Random Sampling": "#8c8c8c", "Genetic Algorithm": "#2e8b57",
    "Particle Swarm": "#7b52ab", "Differential Evolution": "#00868b",
    "Simulated Annealing": "#a0522d",
}
MARKERS = {ENUMERATION: "*", PROPOSED: "o", "Classical MVO": "s",
           "Strong Greedy": "^", "Random Sampling": "v",
           "Genetic Algorithm": "D", "Particle Swarm": "P",
           "Differential Evolution": "X", "Simulated Annealing": "<"}


def save_figure(fig: plt.Figure, name: str) -> None:
    for ext in ("png", "pdf"):
        fig.savefig(OUTDIR / "figures" / f"{name}.{ext}", bbox_inches="tight")
    print(f"  saved figures/{name}.png and .pdf")


def figure1_architecture() -> None:
    fig, ax = plt.subplots(figsize=(10.5, 5.4))
    ax.set_axis_off()
    ax.set_xlim(0, 10.5), ax.set_ylim(0, 5.4)
    ax.grid(False)

    def box(x, y, w, h, title, lines, face, edge):
        ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=face, edgecolor=edge,
                                   linewidth=1.4, zorder=2, joinstyle="round"))
        ax.text(x + w / 2, y + h - 0.28, title, ha="center", va="top",
                fontsize=9.5, fontweight="bold", zorder=3)
        for k, line in enumerate(lines):
            ax.text(x + w / 2, y + h - 0.62 - 0.30 * k, line, ha="center", va="top",
                    fontsize=8.2, zorder=3)

    def arrow(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), zorder=1,
                    arrowprops=dict(arrowstyle="-|>", linewidth=1.3,
                                    color="#444444", shrinkA=2, shrinkB=2))

    box(0.15, 3.5, 2.3, 1.55, "Agent 1\nData Preparation",
        ["synthetic GBM,", "5 regimes", "real walk-forward"], "#eef4fb", "#1f6fb2")
    box(2.75, 3.5, 2.3, 1.55, "Agent 2a\nMarket Analyst",
        ["log returns", "annualised mu, Sigma"], "#eef4fb", "#1f6fb2")
    box(5.35, 3.5, 2.3, 1.55, "Agent 2b\nRisk Assessor",
        ["objective f(S)", "= Sharpe ratio", "evaluation budget"], "#eef4fb", "#1f6fb2")
    box(7.95, 3.5, 2.4, 1.55, "Outputs",
        ["tables, figures,", "consistency checks"], "#f2f2f2", "#666666")
    for x in (2.45, 5.05, 7.65):
        arrow(x, 4.28, x + 0.28, 4.28)

    box(2.75, 0.35, 4.9, 2.55, "Agent 2c  Portfolio Optimizer", [], "#fbf6ee", "#e08a1e")
    box(2.95, 1.95, 4.5, 0.72, "Tier 1  multi-risk spectrum",
        ["QUBO surrogate at q in {0.25, 0.5, 1.0}"], "#ffffff", "#e08a1e")
    box(2.95, 1.15, 4.5, 0.72, "Tier 2  QAOA global sampling",
        ["p=2, 1024 shots, COBYLA, state-vector sim"], "#ffffff", "#e08a1e")
    box(2.95, 0.50, 4.5, 0.58, "Tier 3  classical local refinement", [],
        "#ffffff", "#e08a1e")
    arrow(5.2, 1.95, 5.2, 1.87)
    arrow(5.2, 1.15, 5.2, 1.08)
    arrow(6.5, 3.5, 6.5, 2.90)
    arrow(7.65, 1.60, 9.15, 3.50)
    ax.text(5.25, 5.28, "QuOpt-Agents architecture", ha="center", va="top",
            fontsize=12, fontweight="bold")
    save_figure(fig, "figure1_architecture")
    plt.show()


def figure2_runtime() -> None:
    scale = RESULTS[RESULTS.experiment == "scaleup"]
    if scale.empty:
        return
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    for method, block in scale.groupby("method"):
        block = block.sort_values("n")
        seconds = block["seconds"].to_numpy()
        if np.all(~np.isfinite(seconds)):
            continue
        ax.plot(block["n"], seconds, marker=MARKERS.get(method, "o"),
                color=PALETTE.get(method, "#444444"), label=method, linewidth=1.6,
                markersize=6)
    ax.axvline(CFG.qaoa.max_qubits + 0.5, color="#b03060", linestyle="--", linewidth=1.2)
    ax.text(CFG.qaoa.max_qubits + 0.7, ax.get_ylim()[1] * 0.35,
            f"exact state-vector\nceiling ({CFG.qaoa.max_qubits} qubits)",
            fontsize=8, color="#b03060", va="top")
    ax.set_yscale("log")
    ax.set_xlabel("Number of assets N (= number of qubits)")
    ax.set_ylabel("Wall-clock time per instance (s, log scale)")
    ax.set_title("Runtime scaling and the simulability ceiling")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8.5)
    save_figure(fig, "figure2_runtime_scaling")
    plt.show()


def figure3_convergence() -> None:
    if not TRAJECTORIES:
        return
    instance = sorted({inst for _, inst in TRAJECTORIES})[0]
    fig, ax = plt.subplots(figsize=(7.6, 4.4))
    for (method, inst), curve in TRAJECTORIES.items():
        if inst != instance or curve is None or np.all(np.isnan(curve)):
            continue
        x = np.arange(1, curve.size + 1)
        ax.step(x, curve, where="post", label=method, linewidth=1.7,
                color=PALETTE.get(method, "#444444"))
    optimum = RESULTS[(RESULTS.instance == instance)
                      & (RESULTS.method == ENUMERATION)]["sharpe_is"]
    if not optimum.empty:
        ax.axhline(optimum.iloc[0], color="#111111", linestyle="--", linewidth=1.3,
                   label="exact optimum")
    ax.set_xscale("log")
    ax.set_xlabel("Cumulative objective evaluations (log scale)")
    ax.set_ylabel("Best Sharpe ratio found so far")
    ax.set_title(f"Convergence on scenario {instance}, common evaluation budget")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8.5)
    save_figure(fig, "figure3_convergence")
    plt.show()


def figure4_risk_return() -> None:
    sub = RESULTS[(RESULTS.experiment == "synthetic")
                  & RESULTS.method.isin([PROPOSED, "Classical MVO", "Strong Greedy",
                                         "Random Sampling", ENUMERATION])]
    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    for method, block in sub.groupby("method"):
        ax.scatter(block["vol_is"], block["ret_is"], s=46 if method != ENUMERATION else 130,
                   marker=MARKERS.get(method, "o"),
                   facecolor="none" if method == ENUMERATION else PALETTE.get(method),
                   edgecolor=PALETTE.get(method, "#444444"), linewidth=1.2,
                   alpha=0.85, label=method)
    ax.set_xlabel("Portfolio volatility (annualised)")
    ax.set_ylabel("Portfolio expected return (annualised)")
    ax.set_title("Risk-return profile of selected portfolios (in-sample)")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8.5,
              title="Method", title_fontsize=9)
    save_figure(fig, "figure4_risk_return")
    plt.show()


def figure5_distributions() -> None:
    wide = pivot_metric(RESULTS, "synthetic", "sharpe_is")
    methods = [m for m in METHOD_ORDER if m in wide.columns]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 4.5),
                                   gridspec_kw={"width_ratios": [1.35, 1]})

    data = [wide[m].dropna().to_numpy() for m in methods]
    parts = ax1.boxplot(data, vert=True, patch_artist=True, widths=0.6,
                        medianprops=dict(color="#111111", linewidth=1.4),
                        flierprops=dict(marker=".", markersize=4, alpha=0.6))
    for patch, method in zip(parts["boxes"], methods):
        patch.set_facecolor(PALETTE.get(method, "#999999"))
        patch.set_alpha(0.45)
    for k, (method, values) in enumerate(zip(methods, data), start=1):
        jitter = np.random.default_rng(k).normal(0, 0.055, values.size)
        ax1.scatter(np.full(values.size, k) + jitter, values, s=8, alpha=0.5,
                    color=PALETTE.get(method, "#444444"), zorder=3)
    ax1.set_xticks(range(1, len(methods) + 1))
    ax1.set_xticklabels([m.replace(" ", "\n") for m in methods], fontsize=7.6)
    ax1.set_ylabel("In-sample Sharpe ratio")
    ax1.set_title("Sharpe ratio distribution by method")

    if PROPOSED in wide.columns and "Strong Greedy" in wide.columns:
        diff = (wide[PROPOSED] - wide["Strong Greedy"]).dropna().to_numpy()
        ax2.hist(diff, bins=max(8, int(np.sqrt(diff.size)) * 2),
                 color="#1f6fb2", alpha=0.75, edgecolor="white")
        ax2.axvline(0, color="#111111", linestyle="--", linewidth=1.3)
        ax2.axvline(diff.mean(), color="#b03060", linewidth=1.6,
                    label=f"mean {diff.mean():+.2e}")
        ax2.set_xlabel(f"Paired difference: {PROPOSED} minus Strong Greedy")
        ax2.set_ylabel("Scenarios")
        ax2.set_title("Paired differences, the quantity the test uses")
        ax2.legend(fontsize=8.5)
    save_figure(fig, "figure5_sharpe_distributions")
    plt.show()


def figure6_optimality_gap() -> None:
    table, _ = TABLES["table03_aggregate_synthetic"]
    block = table[table["method"] != ENUMERATION].dropna(subset=["gap_mean_pct"])
    if block.empty:
        return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.3))
    order = block.sort_values("gap_mean_pct")
    colors = [PALETTE.get(m, "#666666") for m in order["method"]]
    ax1.barh(order["method"], order["gap_mean_pct"], color=colors, alpha=0.85)
    ax1.set_xlabel("Mean optimality gap (% of optimal Sharpe)")
    ax1.set_title("Distance from the exact optimum")
    ax1.invert_yaxis()
    ax2.barh(order["method"], order["pct_optimal"], color=colors, alpha=0.85)
    ax2.set_xlabel("Scenarios reaching the exact optimum (%)")
    ax2.set_xlim(0, 100)
    ax2.set_title("Frequency of finding the optimum")
    ax2.invert_yaxis()
    save_figure(fig, "figure6_optimality_gap")
    plt.show()


for _fn in (figure1_architecture, figure2_runtime, figure3_convergence,
            figure4_risk_return, figure5_distributions, figure6_optimality_gap):
    print(f"building {_fn.__name__} ...")
    _fn()


## 16. Consistency checks (Editor 2, Comments 2 and 7)

The inconsistencies the reviewers found — a Wilcoxon statistic outside its own range, figure
means that disagreed with the corresponding table, improvement percentages that did not
reconcile — were all detectable mechanically. They survived because nothing checked them.

This section asserts identities that must hold if the results are internally coherent, and
**fails loudly** if any is violated. It runs last, so a rerun of the whole notebook either ends
with `ALL CHECKS PASSED` or names the broken identity. Nothing should be copied into the
manuscript from a run that did not pass.

The checks are deliberately of the kind that catch transcription and formula errors rather
than modelling disagreements:

- no method can beat exhaustive enumeration, so every optimality gap must be non-negative;
- enumeration must itself show a zero gap on every instance;
- every selection must hold exactly `B` assets;
- no searcher may exceed its evaluation budget;
- `t` must equal `mean(diff) / (sd(diff)/sqrt(n))` for the paired test as reported;
- `W` must lie in `[0, n_eff(n_eff+1)/2]`;
- wins, losses and ties must sum to the number of pairs;
- the regime-weighted mean of the per-regime means must equal the pooled mean;
- removing Tier 3 cannot improve the in-sample objective, since Tier 3 only accepts
  improving moves;
- instances above the qubit ceiling must be recorded as not simulable, with no Sharpe value.


In [ ]:
CHECKS: List[Tuple[str, bool, str]] = []


def check(name: str, condition: bool, detail: str = "") -> None:
    CHECKS.append((name, bool(condition), detail))


# 1 - nothing beats the exact optimum
_gaps = RESULTS["gap_pct"].dropna()
check("optimality gaps are non-negative", bool((_gaps >= -1e-7).all()),
      f"min gap {_gaps.min():+.3e} % over {_gaps.size} rows")

# 2 - enumeration is itself optimal everywhere it ran
_enum = RESULTS[(RESULTS.method == ENUMERATION) & RESULTS.gap_pct.notna()]
check("enumeration shows a zero gap", bool((_enum["gap_pct"].abs() < 1e-9).all()),
      f"max |gap| {_enum['gap_pct'].abs().max():.3e} % over {len(_enum)} instances")

# 3 - cardinality is respected on every row that produced a portfolio
_sel = RESULTS[RESULTS["selection"].astype(str) != ""]
_bad = _sel[[len(str(s).split(",")) != int(b)
             for s, b in zip(_sel["selection"], _sel["B"])]]
check("every selection holds exactly B assets", _bad.empty,
      f"{len(_bad)} violations out of {len(_sel)} rows")

# 4 - no searcher exceeded its budget
_stochastic = RESULTS[RESULTS.method.isin(
    ["Random Sampling", "Genetic Algorithm", "Particle Swarm",
     "Differential Evolution", "Simulated Annealing", PROPOSED])]
check("evaluation budget respected",
      bool((_stochastic["n_evals"] <= CFG.eval_budget).all()),
      f"max {_stochastic['n_evals'].max():,.0f} of {CFG.eval_budget:,} allowed")

# 5 - the reported t statistic is the paired t statistic
_t_errors = []
for _, r in STATS_IS.iterrows():
    if r["n"] > 1 and r["sd_diff"] > TIE_TOL:
        implied = r["mean_diff"] / (r["sd_diff"] / math.sqrt(r["n"]))
        _t_errors.append(abs(implied - r["t_stat"]))
check("t = mean(diff) / (sd(diff)/sqrt(n))",
      (not _t_errors) or max(_t_errors) < 1e-6,
      f"max deviation {max(_t_errors):.2e}" if _t_errors else "no untied comparisons")

# 6 - Wilcoxon statistic inside its definitional bound
_w = pd.concat([STATS_IS, STATS_OOS, STATS_REAL]).dropna(subset=["W"])
check("0 <= W <= n_eff(n_eff+1)/2",
      bool(((_w["W"] >= 0) & (_w["W"] <= _w["W_max"] + 1e-9)).all()),
      f"{len(_w)} comparisons checked")

# 7 - win / loss / tie decomposition is complete
check("wins + losses + ties = n",
      bool((STATS_IS["wins"] + STATS_IS["losses"] + STATS_IS["ties"]
            == STATS_IS["n"]).all()),
      f"{len(STATS_IS)} comparisons")

# 8 - regime-weighted mean reconciles with the pooled mean
_syn = RESULTS[(RESULTS.experiment == "synthetic") & (RESULTS.method == PROPOSED)]
_by_regime = _syn.groupby("regime")["sharpe_is"].agg(["mean", "size"])
_weighted = float((_by_regime["mean"] * _by_regime["size"]).sum() / _by_regime["size"].sum())
_pooled = float(_syn["sharpe_is"].mean())
check("regime-weighted mean equals pooled mean", abs(_weighted - _pooled) < 1e-9,
      f"weighted {_weighted:.8f} vs pooled {_pooled:.8f}")

# 9 - Tier 3 cannot hurt the in-sample objective
_ta = RESULTS[RESULTS.experiment == "tier_ablation"]
if not _ta.empty:
    _full = _ta[_ta.method == "Full (T1+T2+T3)"]["sharpe_is"].mean()
    _no_t3 = _ta[_ta.method == "No Tier 3 (raw QAOA)"]["sharpe_is"].mean()
    check("removing Tier 3 does not improve the objective", _no_t3 <= _full + 1e-9,
          f"full {_full:.6f} vs no-Tier-3 {_no_t3:.6f}")

# 10 - the qubit ceiling is reported, not silently bypassed
_scale = RESULTS[(RESULTS.experiment == "scaleup") & (RESULTS.method == PROPOSED)]
_over = _scale[_scale["n"] > CFG.qaoa.max_qubits]
check("instances above the qubit ceiling are marked not simulable",
      _over.empty or bool(_over["selection"].astype(str).eq("").all()),
      f"{len(_over)} instances above {CFG.qaoa.max_qubits} qubits, "
      f"all reported as not simulable")

print("=" * 92)
for _name, _ok, _detail in CHECKS:
    print(f"{'PASS' if _ok else 'FAIL'}  {_name:<52} {_detail}")
print("=" * 92)
_failed = [c for c in CHECKS if not c[1]]
if _failed:
    raise AssertionError("Consistency checks failed: "
                         + "; ".join(f"{n} ({d})" for n, _, d in _failed))
print(f"ALL CHECKS PASSED  ({len(CHECKS)} identities verified)")

# --- reproducibility record ---------------------------------------------------
MANIFEST = {
    "config": {k: (list(v) if isinstance(v, tuple) else v)
               for k, v in asdict(CFG).items()},
    "environment": ENV,
    "qiskit_cross_check": QISKIT_INFO,
    "real_data_is_synthetic_surrogate": REAL_DATA_IS_SYNTHETIC,
    "quick_mode": QUICK,
    "n_result_rows": int(len(RESULTS)),
    "checks_passed": len(CHECKS),
    "enumeration_limit": ENUM_LIMIT,
    "tie_tolerance": TIE_TOL,
}
(OUTDIR / "results" / "manifest.json").write_text(
    json.dumps(MANIFEST, indent=2, default=str), encoding="utf-8")
print(f"\nmanifest written to {OUTDIR / 'results' / 'manifest.json'}")


In [ ]:
agg, _ = TABLES["table03_aggregate_synthetic"]
agg_idx = agg.set_index("method")
tier2, _ = TABLES["table10_tier2_ablation"]
tier2_idx = tier2.set_index("configuration")

n_syn = RESULTS[RESULTS.experiment == "synthetic"]["instance"].nunique()
prop = agg_idx.loc[PROPOSED]
greedy_row = STATS_IS[STATS_IS.comparison.str.endswith("Strong Greedy")]

print("=" * 92)
print("NUMBERS FOR THE REVISED MANUSCRIPT".center(92))
print("=" * 92)

print(f"""
Setup
  N = {CFG.n_assets} assets, B = {CFG.budget}, feasible portfolios = {math.comb(CFG.n_assets, CFG.budget):,}
  {n_syn} synthetic scenarios across {len(REGIMES)} regimes; {len(REAL_WINDOWS)} real walk-forward windows
  every stochastic searcher: {CFG.eval_budget:,} objective evaluations
  QAOA: p = {CFG.qaoa.depth}, {CFG.qaoa.shots} shots, {CFG.qaoa.optimizer}, {CFG.qaoa.n_restarts} restarts

Proposed method, synthetic study
  mean in-sample Sharpe      {prop['sharpe_is_mean']:.4f} (sd {prop['sharpe_is_sd']:.4f})
  mean out-of-sample Sharpe  {prop['sharpe_oos_mean']:.4f} (sd {prop['sharpe_oos_sd']:.4f})
  mean optimality gap        {prop['gap_mean_pct']:.4f} %
  reached the exact optimum  {prop['pct_optimal']:.1f} % of scenarios
  mean objective evaluations {prop['f_evals_mean']:.0f}
  mean wall-clock            {prop['seconds_mean']:.2f} s

How the baselines compare on the optimality gap""")
for _m, _r in agg_idx.iterrows():
    if _m == ENUMERATION or not np.isfinite(_r["gap_mean_pct"]):
        continue
    print(f"  {_m:<24} gap {_r['gap_mean_pct']:>8.4f} %   optimum found "
          f"{_r['pct_optimal']:>5.1f} % of the time")

if not greedy_row.empty:
    g = greedy_row.iloc[0]
    print(f"""
Paired comparison against Strong Greedy (in-sample, n = {int(g['n'])})
  wins / losses / ties       {int(g['wins'])} / {int(g['losses'])} / {int(g['ties'])}
  mean paired difference     {g['mean_diff']:+.6f}
  95% CI on the difference   [{g['ci_diff_lo']:+.6f}, {g['ci_diff_hi']:+.6f}]
  paired t                   {g['t_stat']:.4f}   p = {g['p_ttest']:.4g}
  Wilcoxon W                 {g['W']:.1f}  (bound {g['W_max']:.1f}, n_eff = {int(g['n_eff'])})
  p (Wilcoxon, {g['wilcoxon_method']})       {g['p_wilcoxon']:.4g}
  Cohen's d (paired)         {g['cohens_d']:.4f}
  improvement, ratio of means      {g['improve_ratio_of_means']:+.4f} %
  improvement, mean of ratios      {g['improve_mean_of_ratios']:+.4f} %""")

print("\nTier-2 substitution: what the quantum layer contributes")
for _arm, _r in tier2_idx.iterrows():
    print(f"  {_arm:<32} Sharpe {_r['sharpe_is_mean']:.6f}   "
          f"gap {_r['gap_mean_pct']:>7.4f} %   optimum {_r['pct_optimal']:>5.1f} %")

_qaoa_arm = tier2_idx.loc["Tier-2 = QAOA", "sharpe_is_mean"] \
    if "Tier-2 = QAOA" in tier2_idx.index else float("nan")
_best_classical = tier2_idx.drop(index="Tier-2 = QAOA", errors="ignore")["sharpe_is_mean"].max()
print(f"\n  QAOA arm {_qaoa_arm:.6f} vs best classical stand-in {_best_classical:.6f} "
      f"-> difference {100 * (_qaoa_arm - _best_classical) / abs(_best_classical):+.4f} %")

print("\nSimulability")
_scale_prop = RESULTS[(RESULTS.experiment == "scaleup") & (RESULTS.method == PROPOSED)]
for _, _r in _scale_prop.iterrows():
    _status = "simulated" if str(_r["selection"]) else "NOT simulable"
    print(f"  {_r['instance']:<14} {int(_r['n']):>3} qubits   {_status}")

print("\n" + "=" * 92)
print("CLAIMS IN THE SUBMITTED MANUSCRIPT THAT THIS RUN DOES NOT SUPPORT".center(92))
print("=" * 92)
print("""
  1. Any statement that QAOA outperforms classical baselines at this problem size.
     The Tier-2 ablation above shows the quantum layer is not what produces the result.
  2. The 92% / 98% win rates. They came from a comparison in which the proposed method
     searched a superset of the greedy baseline's single candidate. Replace with the
     optimality-gap and win/loss/tie figures above.
  3. Any QAOA hyperparameter description attached to the submitted code, which ran an exact
     eigensolver rather than QAOA. Use the Section 5 specification instead.
  4. Improvement percentages quoted without saying which estimator produced them. Both are
     printed above; pick one, define it in the caption, and use it consistently.
  5. In-sample Sharpe reported as performance. Use the out-of-sample columns, or state
     explicitly that the figure is in-sample.
""")
print("=" * 92)
